[![](imagens/colab-badge.png){height="20px" style="height:20px;vertical-align:middle"}](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.pt/cap03/cap03_aluno.ipynb)
[![](imagens/github-badge.png){height="20px" style="height:20px;vertical-align:middle"}](https://github.com/fzampirolli/pdi-vc)

# Operações Espaciais: Intensidade, Histograma e Filtragem

Este capítulo aprofunda o processamento de imagens no domínio espacial, partindo da manipulação direta de pixels e histogramas para o realce de contraste, até a aplicação de filtros locais por convolução para suavização, redução de ruído e detecção de bordas. O objetivo é desenvolver a intuição matemática e computacional que sustenta grande parte dos algoritmos modernos de Visão Computacional.

## Objetivos

Ao final deste capítulo, você será capaz de:

* **Manipular intensidade e pixels:** Executar operações aritméticas saturadas (`mm.addm`, `mm.subm`) e lógicas bit a bit (`mm.band`, `mm.bor`, `mm.bnot`) para combinação e seleção de regiões de interesse (ROI), e aplicar *alpha blending* (`mm.blend`) para fusão ponderada de imagens;
* **Processar histogramas:** Interpretar o histograma como diagnóstico tonal e aplicar equalização global via CDF (`mm.equalize`); na trilha Python, também a equalização adaptativa (CLAHE) e a especificação de histograma para transferência de perfil tonal entre imagens;
* **Compreender fundamentos espaciais:** Entender vizinhança, *padding* de borda (`mm.pad`) e a diferença entre correlação cruzada (`mm.conv`) e convolução — incluindo por que *kernels* assimétricos como o de Sobel produzem resultados distintos nas duas operações;
* **Aplicar filtragem de suavização:** Usar o filtro de média (`mm.blur`, ou `mm.conv` com *kernel* uniforme) e o filtro Gaussiano (`mm.gaussian`) para redução de ruído, compreendendo a vantagem da ponderação radial e da separabilidade Gaussiana;
* **Aplicar filtragem de realce:** Usar o Laplaciano $w_4$ e $w_8$ (`mm.laplacian`) para realce isotrópico de bordas, o operador de Sobel (`mm.sobel`) para a magnitude do gradiente — e, na trilha Python, a decomposição direcional $G_x$, $G_y$ e ângulo —, e o *Unsharp Masking* (`mm.usm`) para amplificação de alta frequência controlada pelo parâmetro $k$;
* **Utilizar filtros de ordem:** Aplicar o filtro da mediana (`mm.median`) para remoção de ruído sal e pimenta, compreendendo por que sua natureza não linear e a robustez a *outliers* o tornam superior aos filtros lineares nesse cenário;
* **Resolver problemas práticos:** Encadear técnicas em *pipelines* de pré-processamento (equalização → Gaussiano → Canny; com CLAHE no lugar da equalização na trilha Python) e usar as funções da `morph` (`mm.conv`, `mm.histImg`, `mm.equalize`, `mm.drawImgKernel`) para análise e visualização didática de cada etapa.

## Operações em Nível de Intensidade

O nível mais elementar de processamento de imagens atua diretamente sobre os valores dos pixels, sem considerar vizinhança. Essas operações — chamadas de **transformações de ponto** (*point operations*) — são as mais rápidas computacionalmente e formam a base para técnicas mais complexas.

Formalmente, uma transformação de ponto pode ser descrita como:

$$
g(x,y) = T[f(x,y)]
$$ {#eq-03-ponto}

onde $f(x,y)$ é a imagem de entrada, $g(x,y)$ é a saída e $T$ é uma função aplicada a cada pixel individualmente.

### Preparando o Ambiente Prático

O bloco a seguir carrega a biblioteca `morph` do repositório (o módulo `morph.py` e, na trilha C++, também a `morph.hpp` usada no `#include` das células compiladas).

In [ ]:
#[py]#
#| quarto-raw: true
import os, urllib.request

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup()
from morph import mm
import numpy as np
import cv2


In [ ]:
#[cpp]#
#| quarto-raw: true
import os, urllib.request

os.makedirs("tmp/state", exist_ok=True)  # artefatos de build da trilha C++ (.cpp, binário, PNGs)

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

# O kernel é Python mesmo na trilha C++: `mm` (morph.py) é usado pelos
# simuladores, pela exibição das figuras que o binário C++ gera e pelo
# estado mm::Image entre células. cpp=True baixa também a trilha compilada
# (morph.hpp + stb_image*.h), usada no #include das células %%writefile *.cpp.
import config
config.setup(cpp=True)
from morph import mm
import numpy as np

Como objeto de estudo ao longo deste capítulo, utilizaremos as imagens de vida selvagem apresentadas nas @fig-03-mandrill e @fig-03-leopardo. A partir delas, exploraremos operações espaciais sobre intensidade, histogramas e filtragem, analisando seus efeitos no realce, na suavização, na redução de ruído e na detecção de bordas, de modo a compreender os fundamentos matemáticos e computacionais do PDI.

In [ ]:
#| label: fig-03-mandrill
#| fig-cap: "*Mandrill* (*Mandrillus sphinx*) fotografado em ambiente natural na África do Sul. Crédito: Carlos Guilherme Rodrigues (CC BY-SA 3.0)."
#| echo: true

import os

base    = "https://upload.wikimedia.org/wikipedia/commons"
arquivo = "Carlos_Guilherme_Rodrigues_%2876515283%29.jpeg"
url     = f"{base}/9/9b/{arquivo}"
caminho = "imagens/mandrill-exif.jpg"

if not os.path.exists(caminho):
    os.makedirs("imagens", exist_ok=True)
    mm.write(mm.read(url), caminho)

img_color = mm.read(caminho)
img_gray  = mm.gray(img_color)

mm.show(img_color)

### Operações Aritméticas

Operações aritméticas entre imagens são amplamente usadas em PDI para combinar, comparar ou realçar informações. A **subtração de imagens** é especialmente poderosa para detectar diferenças entre dois quadros — por exemplo, na remoção de fundo estático em câmeras de vigilância:

$$
g(x,y) = f_1(x,y) - f_2(x,y)
$$ {#eq-03-subtracao}

A **adição saturada** limita o resultado ao intervalo $[0, 255]$: valores acima de 255 são fixados em 255, evitando o *overflow* silencioso do tipo `uint8` (ex.: $200 + 100 = 44$ em vez de 300). A **subtração saturada** aplica o mesmo princípio pelo lado inferior: valores negativos são fixados em 0.

::: {.callout-warning}
### Saturação e *overflow* {.unnumbered}
Operações aritméticas em `uint8` sofrem *overflow* silencioso: $200 + 100 = 44$ (não 300). `mm.addm` e `mm.subm` fazem a **saturação automática**, fixando o resultado em $[0, 255]$. O *blending* usa pesos fracionários: `mm.blend` opera internamente em ponto flutuante e só então arredonda e satura para `uint8`.
:::

A @fig-03-aritmetica demonstra adição de uma constante (clareamento) e subtração de uma constante (escurecimento com saturação em 0).

In [ ]:
#| label: fig-03-aritmetica
#| fig-cap: "Operações aritméticas saturadas: adição de constante (clareamento) e subtração de constante (escurecimento com saturação em 0)."
#| echo: true
#| output: true

fundo = 60

img_add = mm.addm(img_gray, fundo)
img_sub = mm.subm(img_gray, fundo)

mm.show(
    [img_gray, img_add, img_sub],
    titles=["Original", "addm (+60)", "subm (−60)"],
    cols=3
)

### Mistura Ponderada (*Alpha Blending*)

A **mistura ponderada** (*alpha blending*) combina duas imagens utilizando pesos complementares $\alpha$ e $(1-\alpha)$:

$$
g(x,y) = \alpha\,f_1(x,y) + (1-\alpha)\,f_2(x,y), \quad \alpha \in [0,1]
$$ {#eq-03-blend}

Quando $\alpha = 1$, obtém-se apenas a imagem $f_1$; quando $\alpha = 0$, apenas $f_2$. Valores intermediários produzem uma transição suave entre ambas, sendo amplamente utilizados em composição de imagens, sobreposição de camadas, marcas d'água e efeitos de fusão visual.

Para que a combinação produza um resultado coerente, é necessário alinhar previamente as regiões de interesse. Na @fig-03-blend, recorta-se o rosto do leopardo com `mm.crop(img_leop_gray, 250, H-300, 100, W-200)` e a região facial do mandril com `mm.crop(img_gray, 100, 400, 380, 530)`, de modo que olhos e estrutura facial fiquem aproximadamente alinhados. O recorte do leopardo é então redimensionado (`mm.resize`) para as dimensões do mandril antes da mistura.

`mm.blend` faz a operação em ponto flutuante — evitando *overflow* nas contas com pesos fracionários — e só então arredonda e satura o resultado para `uint8`.

In [ ]:
#| label: fig-03-leopardo
#| fig-cap: "Retrato de um leopardo (*Panthera pardus*) em ambiente natural. Crédito: C. Brück (CC BY-SA 4.0)."
#| echo: true

import os

base    = "https://upload.wikimedia.org/wikipedia/commons"
arquivo = "Leopard_%28Panthera_pardus%29_portrait.jpg"
url     = f"{base}/9/92/{arquivo}"
caminho = "imagens/leopardo.jpg"

if not os.path.exists(caminho):
    os.makedirs("imagens", exist_ok=True)
    mm.write(mm.read(url), caminho)

img_leop      = mm.read(caminho)
img_leop_gray = mm.gray(img_leop)

mm.show(img_leop)

In [ ]:
#| label: fig-03-blend
#| fig-cap: "*Alpha blending* entre recortes alinhados de mandrill e do leopardo (@fig-03-leopardo) para diferentes valores de α. Em α=1 vê-se apenas mandrill; em α=0, apenas o leopardo; valores intermediários fundem os olhares das duas imagens proporcionalmente."
#| echo: true
#| output: true

# recortes alinhados: rosto do leopardo e região facial do mandril
leo      = mm.crop(img_leop_gray, 250, img_leop_gray.shape[0] - 300, 100, img_leop_gray.shape[1] - 200)
mandrill = mm.crop(img_gray, 100, 400, 380, 530)
leo_r    = mm.resize(leo, (mandrill.shape[1], mandrill.shape[0]), method="bilinear")

mm.show(
    [mm.blend(mandrill, leo_r, alpha=1.0), mm.blend(mandrill, leo_r, alpha=0.8),
     mm.blend(mandrill, leo_r, alpha=0.6), mm.blend(mandrill, leo_r, alpha=0.4),
     mm.blend(mandrill, leo_r, alpha=0.2), mm.blend(mandrill, leo_r, alpha=0.0)],
    titles=["α=1.0", "α=0.8", "α=0.6", "α=0.4", "α=0.2", "α=0.0"],
    cols=6
)

### Operações Lógicas e Máscaras Bit a Bit

As operações lógicas bit a bit (AND, OR e NOT) atuam diretamente sobre os bits de cada pixel e são a base para criação e aplicação de **máscaras** (*masks*) — imagens binárias com apenas 0 (preto) e 255 (branco) usadas para isolar **Regiões de Interesse** (**ROI**).

O comportamento de cada operação decorre da representação binária do 255 (`11111111`) e do 0 (`00000000`):

- **AND** com a máscara: onde $m = 255$, os bits originais são preservados; onde $m = 0$, o pixel é zerado. Resultado: recorte da ROI.
$$g(x,y) = f(x,y) \;\text{AND}\; m(x,y)
$$ {#eq-03-mascara}
- **OR** com a máscara: onde $m = 255$, o pixel é forçado a branco; onde $m = 0$, o valor original é mantido. Resultado: iluminação da ROI.
- **NOT** (sem máscara): inverte todos os bits ($g = 255 - f$), produzindo o negativo fotográfico da imagem.

A @fig-03-logica ilustra as três operações aplicadas à imagem do mandrill com uma máscara circular.

In [ ]:
#| label: fig-03-logica
#| fig-cap: "Operações lógicas bit a bit com máscara circular: AND (isolamento da ROI), OR (iluminação da ROI) e NOT (negativo)."
#| echo: true
#| output: true

h, w = img_gray.shape

# Máscara circular preenchida, centrada na imagem (mm.circle desenha o
# disco; a versão didática, teste de raio pixel a pixel, é mm.circle0).
mask_circ = mm.circle(np.zeros((h, w), dtype=np.uint8),
                      (w // 2, h // 2), min(h, w) // 3 - 10, 255, -1)

# Operações via morph
img_not = mm.bnot(img_gray)            # NOT: negativo fotográfico
img_and = mm.band(img_gray, mask_circ) # preserva apenas a ROI circular
img_or  = mm.bor(img_gray, mask_circ)  # ilumina a região da máscara

mm.show(
    [img_gray, img_and, img_or, img_not],
    titles=["Original", "AND (ROI circular)", "OR (ilumina ROI)", "NOT (negativo)"],
    cols=4
)

## Histograma de Imagens

O **histograma** de uma imagem em tons de cinza é uma função discreta que descreve a distribuição de frequências das intensidades:

$$
h(r_k) = n_k, \quad k = 0, 1, \ldots, L-1
$$ {#eq-03-histograma}

onde $r_k$ é o $k$-ésimo nível de intensidade, $n_k$ é o número de pixels com essa intensidade e $L$ é o total de níveis (tipicamente 256 para 8 bits). O histograma normalizado estima a probabilidade de cada nível:

$$
p(r_k) = \frac{n_k}{MN}
$$ {#eq-03-hist-norm}

onde $MN$ é o total de pixels. Por ser uma **estatística global**, o histograma não carrega informação posicional, mas revela características essenciais como brilho médio, contraste e distribuição tonal. Na prática, `mm.hist(img)` retorna o vetor de contagens $h(r_k)$, que serve tanto para visualização (via `mm.histImg`) quanto para cálculos como **função de distribuição acumulada (CDF)** e equalização.

::: {.callout-note}
### Interpretação do Histograma {.unnumbered}
- **Estreito à esquerda:** imagem subexposta (escura).
- **Estreito à direita:** imagem superexposta (clara).
- **Concentrado no centro:** baixo contraste.
- **Distribuído por toda a faixa:** alto contraste, boa utilização dos tons disponíveis.
:::

A @fig-03-histograma apresenta o histograma da imagem do mandrill, bem como versões escurecida (`mm.subm`) e clareada (`mm.addm`). Observa-se o deslocamento da distribuição de intensidades para a esquerda e para a direita, respectivamente. Note que o intervalo representado no eixo $x$ não corresponde necessariamente a toda a faixa de 0 a 255.

In [ ]:
#| label: fig-03-histograma
#| fig-cap: "Histogramas da imagem original, de uma versão escurecida (−80) e de uma clareada (+80). A subtração/adição satura em 0 e 255."
#| echo: true
#| output: true

img_dark = mm.subm(img_gray, 80)
img_high = mm.addm(img_gray, 80)

mm.show(
    [img_gray, img_dark, img_high,
     mm.histImg(img_gray), mm.histImg(img_dark), mm.histImg(img_high)],
    titles=["Original", "Escurecida (-80)", "Clareada (+80)",
            "Histograma - original", "Histograma - escurecida", "Histograma - clareada"],
    cols=3
)

### Equalização de Histograma

A **equalização de histograma** redistribui as intensidades para que o histograma resultante seja o mais uniforme possível. O mapeamento é dado pela **função de distribuição acumulada (CDF)**:

$$
s_k = T(r_k) = (L-1)\sum_{j=0}^{k} p(r_j) = \frac{L-1}{MN}\sum_{j=0}^{k} n_j
$$ {#eq-03-equalizacao}

A transformação é monotônica: níveis frequentes recebem intervalos maiores no domínio de saída (maior separação → mais contraste), enquanto níveis raros são comprimidos.

O algoritmo completo, em cinco etapas, é apresentado na @tbl-03-equalizacao.

| Etapa | Operação | Fórmula |
|:-----:|:---------|:--------|
| 1 | **Histograma** | $h[k] \leftarrow$ número de pixels com intensidade $k$, $k=0\ldots L-1$ |
| 2 | **Probabilidade** | $p[k] \leftarrow h[k] / MN$ |
| 3 | **CDF** | $\text{cdf}[k] \leftarrow \sum_{j=0}^{k} p[j]$ (soma acumulada) |
| 4 | ***Look-Up Table* (mapeamento)** | $\text{lut}[k] \leftarrow \text{round}(\text{cdf}[k] \times (L-1))$ |
| 5 | **Aplicação** | $g[i,j] \leftarrow \text{lut}[f[i,j]]$ (para todo pixel) |

: Algoritmo de equalização de histograma. {#tbl-03-equalizacao}

Note na @fig-03-equalizacao-didatica que a equalização **redistribui** os tons existentes para posições mais espaçadas na faixa $[0, L-1]$, mas não cria novos tons — a imagem equalizada continua com exatamente 3 tons distintos, agora em $\{1, 5, 7\}$ em vez de $\{2, 3, 4\}$.

In [ ]:
#[py]#
#| label: fig-03-equalizacao-didatica
#| fig-cap: "Equalização de histograma em imagem 5×5 com 3 bits (L=8): imagem original com tons concentrados em {2,3,4}, imagem equalizada com tons redistribuídos para {1,5,7}, e os respectivos histogramas evidenciando o espalhamento das frequências."
#| echo: true
#| output: true

img5 = np.array([[3, 4, 2, 3, 4],
                 [4, 3, 3, 4, 3],
                 [2, 3, 4, 3, 2],
                 [3, 4, 3, 2, 3],
                 [4, 3, 2, 3, 4]], dtype=np.uint8)
L = 8  # 3 bits: níveis 0..7

# 1. Histograma com tamanho garantido até L
h = mm.hist(img5, 3)  # B=3 → 2³=8 níveis, tamanho garantido

p   = h / h.sum()  # 2. Probabilidade de cada nível p[k] = h[k] / MN

cdf = np.cumsum(p) # 3. CDF normalizada ou
# cdf = np.zeros(len(p))
# cdf[0] = p[0]
# for k in range(1, len(p)):
#     cdf[k] = cdf[k-1] + p[k] # cdf[k] = Σ p[j], j=0..k

lut = np.round(cdf * (L - 1)).astype(np.uint8)  # 4. LUT: mapeamento para [0, L-1]

img5_eq = lut[img5] # 5. Aplica LUT pixel a pixel ou, equivalente a:
# l, c = img5.shape
# img5_eq = np.zeros((l, c), dtype=np.uint8)
# for i in range(l):
#     for j in range(c):
#         img5_eq[i, j] = lut[img5[i, j]]      # g[i,j] = lut[f[i,j]]

print("Imagem original (5×5, 3 bits):")
print(img5)
print()
header = f"{'k':>3} {'h[k]':>6} {'p[k]':>7} {'CDF[k]':>8} {'lut[k]':>7}"
print(header)
print("-" * len(header))
for k in range(L):
    print(f"{k:>3} {h[k]:>6} {p[k]:>7.4f} {cdf[k]:>8.4f} {lut[k]:>7}")
print()
print("Imagem equalizada (5×5):")
print(img5_eq)

# Conta tons distintos
tons_orig = len(np.unique(img5))
tons_eq   = len(np.unique(img5_eq))

mm.show(
    [img5, img5_eq, mm.histImg(img5, L-1), mm.histImg(img5_eq, L-1)],
    titles=[f"Original ({tons_orig} tons: {sorted(np.unique(img5).tolist())})",
            f"Equalizada ({tons_eq} tons: {sorted(np.unique(img5_eq).tolist())})",
            "Histograma — Original",
            "Histograma — Equalizada"],
    rows=2, cols=2, figsize=(5, 4), dpi=100
)

In [ ]:
#[cpp]#
#| label: fig-03-equalizacao-didatica
#| fig-cap: "Equalização de histograma numa imagem 5×5 de 3 bits (L=8): tons concentrados em {2,3,4} são redistribuídos pela CDF. `mm::equalize(img, 3)` faz o mapeamento."
#| echo: true
#| output: true

img5 = np.array([[3, 4, 2, 3, 4],
                 [4, 3, 3, 4, 3],
                 [2, 3, 4, 3, 2],
                 [3, 4, 3, 2, 3],
                 [4, 3, 2, 3, 4]], dtype=np.uint8)

img5_eq = mm.equalize(img5, 3)   # L = 2^3 = 8

print("Imagem original 5x5 (3 bits):")
print(mm.drawImg(img5))
print("Imagem equalizada 5x5:")
print(mm.drawImg(img5_eq))

mm.show([img5, img5_eq, mm.histImg(img5), mm.histImg(img5_eq)],
        titles=["Original", "Equalizada", "Histograma - original", "Histograma - equalizada"],
        cols=2)

**Limitação:** a equalização global pode super-realçar ruídos e produzir contraste excessivo em regiões homogêneas. O **CLAHE** (*Contrast Limited Adaptive Histogram Equalization*) reduz esse problema ao aplicar a equalização em blocos locais (*tiles*) e limitar a altura dos picos do histograma antes da equalização.

A @fig-03-equalizacao compara a imagem original, a equalização global via `mm.equalize` e o CLAHE do OpenCV, exibindo também os histogramas resultantes. Diferentemente da equalização global, que utiliza uma única transformação baseada na CDF de toda a imagem, o CLAHE adapta o contraste a cada região, sendo particularmente útil em imagens com iluminação não uniforme.

No exemplo, foi utilizado `clipLimit=2.0` e `tileGridSize=(32,32)`. O parâmetro `clipLimit` define o quanto os picos do histograma local podem crescer antes de serem cortados (*clipped*). No OpenCV, esse valor é um fator relativo: o limite real é aproximadamente calculado como `clipLimit × (número de pixels do bloco / número de níveis de cinza)`. Por exemplo, em um bloco com 4096 pixels e uma imagem de 8 bits (256 níveis de cinza), a frequência média por nível é $4096/256=16$. Assim, `clipLimit=2.0` permite picos de aproximadamente $2\times16=32$ ocorrências antes do corte. As ocorrências excedentes não são descartadas: elas são redistribuídas entre os demais níveis de cinza do histograma, reduzindo a concentração excessiva em poucos níveis e evitando uma amplificação exagerada do contraste local. Valores menores limitam mais o contraste e reduzem a amplificação de ruído, enquanto valores maiores permitem um realce mais intenso, mas podem introduzir artefatos.

- `clipLimit=1.0`: realce suave e conservador;
- `clipLimit=2.0`: bom equilíbrio entre contraste e naturalidade;
- `clipLimit=4.0`: maior destaque para detalhes locais;
- `clipLimit=8.0`: contraste agressivo, com possível amplificação de ruído.

Assim, o CLAHE costuma produzir resultados mais naturais do que a equalização global, especialmente em imagens com sombras, reflexos ou iluminação desigual.

In [ ]:
#[py]#
#| label: fig-03-equalizacao
#| fig-cap: "Equalização de histograma: mm.equalize (global via CDF) vs. CLAHE (adaptativa com limitação de contraste). Os histogramas revelam o espalhamento progressivo das intensidades."
#| echo: true
#| output: true

img_eq    = mm.equalize(img_gray)
clahe     = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(32, 32))
img_clahe = clahe.apply(img_gray)

images = [img_gray, img_eq, img_clahe]
titles = ["Original", "mm.equalize (CDF)", "CLAHE"]

mm.show(
    images + [mm.histImg(img) for img in images],
    titles=titles + [f"Histograma — {t}" for t in titles],
    rows=2, cols=3,
    figsize=(14, 8)
)

In [ ]:
#[cpp]#
#| label: fig-03-equalizacao
#| fig-cap: "Equalização de histograma global (mm::equalize, via CDF) e os histogramas antes/depois. CLAHE (adaptativa) fica só na trilha Python — não tem equivalente em morph.hpp."
#| echo: true
#| output: true

img_eq = mm.equalize(img_gray)

mm.show(
    [img_gray, img_eq, mm.histImg(img_gray), mm.histImg(img_eq)],
    titles=["Original", "mm.equalize (CDF)", "Histograma - original", "Histograma - equalizado"],
    cols=2
)

### Especificação de Histograma

Enquanto a equalização impõe uma distribuição uniforme, a **especificação de histograma** (*histogram matching*) permite que o histograma da imagem de saída siga uma distribuição **arbitrária** — por exemplo, o histograma de outra imagem de referência.

O procedimento envolve três etapas:

1. Calcular a CDF da imagem de entrada: $P_r(r_k)$.
2. Calcular a CDF da imagem de referência: $P_z(z_k)$.
3. Para cada nível $r_k$, encontrar o nível $z$ que minimiza $|P_z(z) - P_r(r_k)|$.

$$
T(r_k) = \arg\min_{z}\,|P_z(z) - P_r(r_k)|
$$ {#eq-03-especificacao}

Na @fig-03-especificacao, transferimos o perfil tonal do leopardo (@fig-03-leopardo) para a imagem do mandrill — uma aplicação direta do conceito visto no *blending*: em vez de fundir pixels, aqui fundimos distribuições tonais.

In [ ]:
#[py]#
#| label: fig-03-especificacao
#| fig-cap: "Especificação de histograma: mandril mapeada para o perfil tonal do leopardo. A CDF da saída aproxima a CDF de referência."
#| echo: true
#| output: true

# mapeia src para o perfil tonal de ref via CDF
def hist_specify(src, ref):
    cdf_src = np.cumsum(mm.hist(src) / src.size)
    cdf_ref = np.cumsum(mm.hist(ref) / ref.size)
    lut = np.array([np.argmin(np.abs(cdf_ref - v)) for v in cdf_src], dtype=np.uint8)
    return lut[src]

img_spec = hist_specify(img_gray, img_leop_gray)

mm.show(
    [img_gray, img_leop_gray, img_spec,
     mm.histImg(img_gray), mm.histImg(img_leop_gray), mm.histImg(img_spec)],
    titles=["Mandrill (original)", "Leopardo (referência)", "Mandrill → perfil do leopardo",
            "Histograma — Mandrill", "Histograma — Leopardo", "Histograma — resultado"],
    rows=2, cols=3, figsize=(12, 9)
)

In [ ]:
#[cpp]#
#| label: fig-03-especificacao
#| fig-cap: "Especificação de histograma (mapear o mandril para o perfil tonal do leopardo) precisa da CDF inversa da referência — fica só na trilha Python. Aqui, a equalização global do mandril, como comparação."
#| echo: true
#| output: true

img_eq = mm.equalize(img_gray)

mm.show([img_gray, img_leop_gray, img_eq],
        titles=["Mandril (original)", "Leopardo (referencia)", "Mandril equalizado"],
        cols=3)

## Fundamentos Espaciais: Vizinhança, Convolução e *Kernels*

As operações de filtragem espacial não atuam em um único pixel isolado, mas em uma **vizinhança** ao seu redor. Para isso, utiliza-se uma pequena matriz de coeficientes denominada **kernel** (ou máscara), que percorre toda a imagem por meio de uma **janela deslizante** (*sliding window*).

As janelas mais comuns são 3×3, 5×5 e 7×7. Em uma janela 3×3, por exemplo, o pixel central é processado juntamente com seus oito vizinhos imediatos. Em cada posição da janela, os valores dos pixels são combinados com os coeficientes do *kernel*, produzindo um novo valor para o pixel central.

### Vizinhança

Considere uma janela 3×3 centrada no pixel $(x,y)$:

$$
\begin{bmatrix}
(x-1,y-1) & (x,y-1) & (x+1,y-1) \\
(x-1,y)   & (x,y)   & (x+1,y)   \\
(x-1,y+1) & (x,y+1) & (x+1,y+1)
\end{bmatrix}
$$ {#eq-03-box3x3}

De forma geral, uma janela de tamanho $(2a+1)\times(2b+1)$ abrange todos os pixels situados até $a$ posições na horizontal e até $b$ posições na vertical em relação ao pixel central. Assim, uma janela 3×3 corresponde a $a=b=1$, uma janela 5×5 a $a=b=2$, e assim por diante.

Matematicamente, a vizinhança é definida por

$$
\mathcal{V}(x,y)=
\{(x+s,\,y+t): -a\le s\le a,\,-b\le t\le b\}
$$ {#eq-03-vizinhanca}

### Tratamento de Bordas

Pixels próximos às bordas possuem parte de sua vizinhança fora da imagem. Para aplicar filtros nessas regiões, é preciso definir como os valores externos serão obtidos. As três estratégias mais comuns (com a constante equivalente do OpenCV entre parênteses) são:

- **Zero-padding** (`BORDER_CONSTANT`): completa a região externa com zeros.
- **Replicação** (`BORDER_REPLICATE`): repete o valor do pixel da borda.
- **Reflexão** (`BORDER_REFLECT_101`): espelha os pixels vizinhos, sem repetir o da borda.

`mm.conv` usa a reflexão por padrão, pois preserva melhor a continuidade dos níveis de cinza e reduz artefatos no tratamento das bordas.

O exemplo a seguir compara as três estratégias com `mm.pad` numa matriz 3×3. Observe como cada uma preenche os pixels externos necessários para aplicar um filtro 3×3 também nos cantos.

In [ ]:
img = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
], dtype=np.uint8)

bordas = ["constant", "replicate", "reflect101"]

print("Imagem original:")
print(mm.drawImg(img))

for k in [3, 5]:
    b = k // 2
    print(f"=== Kernel {k}x{k} (padding b={b}) ===")
    for nome in bordas:
        print(nome)
        print(mm.drawImg(mm.pad(img, b, border=nome)))

<!-- #[py]# -->
Observe que o resultado de um filtro pode variar significativamente conforme o tratamento adotado para as bordas da imagem.

Na biblioteca didática `morph.py`, os algoritmos implementados diretamente em Python (sem OpenCV ou *scikit-image*) — as variantes de sufixo `0`, como `mm.conv0` — não usam *padding*. Para cada pixel, os elementos da vizinhança são examinados individualmente, e apenas os vizinhos cujas coordenadas pertencem ao domínio da imagem participam do cálculo. Assim, nas regiões próximas às bordas a vizinhança efetiva pode conter menos pixels que a janela especificada; nos operadores baseados em `mm._viz` (por exemplo `mm.laplacian`), os pixels de borda com janela incompleta são deixados inalterados.

<!-- #[cpp]# -->
Observe que o resultado de um filtro pode variar significativamente conforme o tratamento adotado para as bordas da imagem.

Na `morph.hpp`, as funções de filtragem (`mm::conv`, `mm::blur`, `mm::gaussian`, `mm::laplacian`, `mm::usm`) aplicam *padding* por **reflexão** (`mm::Border::REFLECT101`) por padrão — o mesmo comportamento do `cv2.filter2D`. A variante didática `mm::conv0` usa `mm::Border::KEEP`: os pixels da borda mantêm o valor original, sem o filtro. Já `mm::sobel` e `mm::prewitt` deixam a borda em zero (calculam apenas o interior).

### Correlação vs. Convolução

Existem dois mecanismos matematicamente relacionados.

**Correlação cruzada** (*cross-correlation*) — o *kernel* é aplicado diretamente:

$$
g(x,y) = \sum_{s=-a}^{a}\sum_{t=-b}^{b} w(s,t)\,f(x+s,\,y+t)
$$ {#eq-03-correlacao}

**Convolução bidimensional** — o *kernel* é rotacionado 180° antes da aplicação:

$$
g(x,y) = \sum_{s=-a}^{a}\sum_{t=-b}^{b} w(s,t)\,f(x-s,\,y-t)
$$ {#eq-03-convolucao}

Para *kernels* simétricos (Gaussiano, Laplaciano, média) as duas operações produzem resultados idênticos. Para *kernels* assimétricos (Sobel, Prewitt) a diferença é significativa, como mostram os exemplos a seguir.

#### Correlação (`mm.conv`)

In [ ]:
# imagem 4x4 (uint8) e kernel assimétrico 3x3
img = np.array([[ 1,  2,  3,  4],
                [ 5,  6,  7,  8],
                [ 9, 10, 11, 12],
                [13, 14, 15, 16]], dtype=np.uint8)

w = np.array([[0, 1, 2],
              [0, 0, 0],
              [0, 0, 0]], dtype=np.float32)

corr = mm.conv(img, w, border="constant")  # zero fora da imagem

print("Imagem original:");         print(mm.drawImg(img))
print("Kernel:");                  print(mm.drawImg(w))
print("Resultado da correlação:"); print(mm.drawImg(corr))

`mm.conv` realiza correlação, isto é, aplica o *kernel* exatamente na orientação fornecida.

#### Convolução

In [ ]:
# repetidos aqui para a célula ser independente
img = np.array([[ 1,  2,  3,  4],
                [ 5,  6,  7,  8],
                [ 9, 10, 11, 12],
                [13, 14, 15, 16]], dtype=np.uint8)
w   = np.array([[0, 1, 2],
                [0, 0, 0],
                [0, 0, 0]], dtype=np.float32)

# kernel rotacionado 180° (equivale a np.rot90(w, 2))
w_conv = np.array([[0, 0, 0],
                   [0, 0, 0],
                   [2, 1, 0]], dtype=np.float32)

conv = mm.conv(img, w_conv, border="constant")

print("Imagem original:");            print(mm.drawImg(img))
print("Kernel original:");            print(mm.drawImg(w))
print("Kernel rotacionado 180°:");    print(mm.drawImg(w_conv))
print("Resultado da convolução:");    print(mm.drawImg(conv))

A convolução utiliza o *kernel* rotacionado em 180°. Para reproduzir a definição matemática de convolução, rotaciona-se o *kernel* (aqui, `[[0,1,2],[0,0,0],[0,0,0]]` → `[[0,0,0],[0,0,0],[2,1,0]]`) antes de aplicar `mm.conv`.

### O Papel do *Kernel*

Os coeficientes do *kernel* determinam completamente o efeito produzido pelo filtro, conforme resumido na @tbl-03-kernels.

| Característica | Efeito típico |
|:---|:---|
| Coeficientes positivos com soma 1 | Suavização (*passa-baixa*) |
| Soma igual a 0, com valores positivos e negativos | Detecção de bordas (*passa-alta*) |
| Coeficiente central positivo dominante e vizinhos negativos | Realce de nitidez |
| Coeficientes assimétricos | Gradiente direcional |

: Interpretação típica dos coeficientes do *kernel*. {#tbl-03-kernels}

**Exemplos:**

Suavização:
$$
\frac{1}{9}
\begin{bmatrix}
1&1&1\\
1&1&1\\
1&1&1
\end{bmatrix}
$$

Detecção de bordas:
$$
\begin{bmatrix}
-1&-1&-1\\
-1&8&-1\\
-1&-1&-1
\end{bmatrix}
$$

Realce de nitidez:
$$
\begin{bmatrix}
0&-1&0\\
-1&5&-1\\
0&-1&0
\end{bmatrix}
$$

Gradiente direcional (Sobel):
$$
\begin{bmatrix}
-1&0&1\\
-2&0&2\\
-1&0&1
\end{bmatrix}
$$

A @fig-03-convolucao-passo demonstra o mecanismo passo a passo: para cada posição da janela, multiplica-se cada coeficiente do *kernel* pelo pixel correspondente da vizinhança e somam-se os produtos obtidos. O resultado é exatamente o valor definido pela @eq-03-correlacao para aquela posição da imagem. Embora as imagens produzidas por `mm.conv0` e `cv2.filter2D` (ou `mm.conv`) sejam visualmente muito semelhantes, a implementação baseada em OpenCV é milhares de vezes mais rápida, como mostrado a seguir.

::: {.callout-warning}

### Desempenho: laços Python vs. operações vetorizadas {.unnumbered}

A função `mm.conv0` implementa a correlação diretamente em Python por meio de laços aninhados. Embora essa abordagem seja adequada para fins didáticos, ela executa um grande número de operações e torna-se lenta para imagens maiores.

Já `mm.conv` utiliza `cv2.filter2D`, implementado em C++ e otimizado para operações matriciais. No exemplo apresentado, a versão vetorizada foi mais de **3000 vezes mais rápida** que a implementação didática, produzindo um resultado visualmente equivalente.

As diferenças numéricas observadas concentram-se principalmente nas bordas da imagem. Em `mm.conv0`, os pixels da borda permanecem inalterados, enquanto `mm.conv` utiliza uma estratégia de reflexão das bordas (`cv2.BORDER_REFLECT_101`, padrão do `cv2.filter2D`).

Por isso, `mm.conv0` deve ser utilizado para compreender o algoritmo, enquanto `mm.conv` é a opção recomendada para aplicações práticas.

:::

In [ ]:
#[py]#
#| label: fig-03-convolucao-passo
#| fig-cap: "Correlação com *kernel* de média 3×3: versão didática (mm.conv0) vs. vetorizada (mm.conv via cv2.filter2D). A diferença máxima de 39 ocorre nas bordas."
#| echo: true
#| output: true

import time

w_mean = np.ones((3,3), dtype=np.float32) / 9.0
img_gray = img_leop_gray
# Demonstração numérica em patch 5×5
patch = img_gray[250:255, 250:255].astype(np.float32)
roi   = patch[0:3, 0:3]
print("Patch 5×5 (intensidades):")
print(patch.astype(np.int32))
print(f"\nKernel 3×3 de média:\n{w_mean}")
print(f"\nCorrelação no pixel central [1,1]: \
      {(w_mean * roi).sum():.1f}  (original: {patch[1,1]:.0f})")

# Comparação de tempo na imagem completa
t0 = time.perf_counter()
img_conv0 = mm.conv0(img_gray, w_mean)
t_conv0 = time.perf_counter() - t0

t0 = time.perf_counter()
img_conv = mm.conv(img_gray, w_mean)
t_conv = time.perf_counter() - t0

diff = np.abs(img_conv0.astype(int) - img_conv.astype(int)).max()
print(f"\nTempos na imagem {img_gray.shape[0]}x{img_gray.shape[1]}:")
print(f"  mm.conv0 (laços Python): {t_conv0:.3f} s")
print(f"  mm.conv  (cv2.filter2D): {t_conv:.5f} s")
print(f"  Aceleração:              {t_conv0/t_conv:.0f}× mais rápido")
print(f"  Diferença máxima:        {diff} (bordas com padding diferente)")

mm.show(
    [img_gray, img_conv0, img_conv],
    titles=["Original", f"conv0 ({t_conv0:.2f}s)", f"conv ({t_conv:.4f}s)"],
    cols=3
)

In [ ]:
#[cpp]#
#| label: fig-03-convolucao-passo
#| fig-cap: "Correlação com *kernel* de média 3×3: versão didática mm::conv0 (bordas preservadas) vs. mm::conv (borda refletida). A diferença se concentra nas bordas."
#| echo: true
#| output: true

w_mean = np.ones((3, 3), dtype=np.float32) / 9.0
img_gray = img_leop_gray

img_conv0 = mm.conv0(img_gray, w_mean)   # laços, bordas preservadas
img_conv  = mm.conv(img_gray, w_mean)    # borda refletida

print("Correlacao no pixel central [251,251]:")
print(f"  original = {img_gray[251, 251]}")
print(f"  conv0    = {img_conv0[251, 251]}")
print(f"  conv     = {img_conv[251, 251]}")

mm.show([img_gray, img_conv0, img_conv],
        titles=["Original", "conv0 (laços)", "conv (vetorizado)"], cols=3)

In [ ]:
#| echo: false
# A partir daqui o "sujeito" dos exemplos de filtragem passa a ser o
# leopardo (mais textura e bordas que o mandril).
img_gray = mm.gray(img_leop)

### Exemplo Numérico: Correlação Passo a Passo

Para tornar concreto o mecanismo da @eq-03-correlacao, considere o *kernel* de média 3×3 ($a=b=1$, todos os coeficientes $= 1/9 \approx 0{,}111$) aplicado ao *patch* 5×5 extraído da imagem do leopardo. A @fig-03-patch exibe o *patch* com grade e destaca em amarelo a janela 3×3 centrada no pixel $[1,1]$:

In [ ]:
#| label: fig-03-patch
#| fig-cap: "*Patch* 5×5 extraído da imagem do leopardo (posição [250:255, 250:255]). A janela amarela destaca a vizinhança 3×3 centrada no pixel [1,1] onde a correlação será calculada."
#| echo: true
#| output: true

patch = img_gray[250:255, 250:255]
B     = np.ones((3, 3), dtype=np.uint8)

print("Patch 5×5 (intensidades):")
print(mm.drawImg(patch))

mm.drawImgKernel(patch, B, 1, 1, 40)

Para ilustrar o cálculo da correlação, considere o pixel na posição $[1,1]$ do *patch* 5×5 mostrado na @fig-03-patch. Essa posição foi escolhida apenas por conveniência didática, pois possui uma vizinhança 3×3 completa ao seu redor.

Os valores dessa vizinhança correspondem à submatriz superior esquerda do *patch*:

$$
\text{vizinhança} = \begin{bmatrix}
 91 &  95 & 108 \\
106 & 107 & 108 \\
102 & 103 & 107
\end{bmatrix}
$$

Aplicando a @eq-03-correlacao com o kernel de média:

$$
g[1,1] = \frac{ 91+95+108+106+107+108+102+103+107}{9} = \frac{927}{9} = 103
$$

O resultado (103) é ligeiramente menor que o valor original do pixel central (107), pois a média incorpora vizinhos de menor intensidade, produzindo o efeito de suavização. Na prática, o algoritmo inicia o processamento em $[0,0]$ e repete esse mesmo cálculo para cada posição da imagem, deslocando a janela até cobrir todo o domínio.

## Filtragem Espacial de Suavização

Os filtros de suavização (*smoothing filters*) atenuam variações bruscas de intensidade, reduzindo ruído e detalhes de alta frequência. São filtros **passa-baixa** — preservam as componentes de baixa frequência (estruturas grandes) e atenuam as de alta frequência (ruído, bordas).

### Filtro de Média (*Box Filter*)

O filtro de média utiliza um *kernel* uniforme de tamanho $n \times n$, onde todos os coeficientes valem $1/n^2$:

$$
w_{\text{média}} = \frac{1}{n^2}
\begin{bmatrix}
1 & \cdots & 1 \\
\vdots & \ddots & \vdots \\
1 & \cdots & 1
\end{bmatrix}_{n \times n}
$$ {#eq-03-media}

Cada pixel de saída é a média aritmética dos $n^2$ pixels de sua vizinhança. Note que a soma dos coeficientes é sempre 1 — o brilho médio da imagem é preservado. *Kernels* maiores produzem suavização mais agressiva, mas borram progressivamente as bordas.

A @fig-03-media mostra o efeito do filtro de média com *kernels* $3\times3$, $7\times7$ e $15\times15$ sobre um detalhe da imagem do leopardo. Os resultados foram obtidos com `mm.blur`, que implementa o filtro de média por meio da função `cv2.blur`, equivalente à convolução da imagem com um *kernel* uniforme cujos coeficientes são $h(x,y)=1/N^2$; de forma equivalente, o mesmo resultado pode ser obtido com `mm.conv`, calculando ($g=f*h$). À medida que o *kernel* aumenta, mais pixels contribuem para cada valor de saída, intensificando a suavização, reduzindo o ruído e tornando detalhes finos e bordas progressivamente mais borrados.


In [ ]:
#| label: fig-03-media
#| fig-cap: "Filtro de média com *kernels* de tamanho crescente (3×3, 7×7, 15×15). O borramento das bordas aumenta com o tamanho do *kernel*."
#| echo: true
#| output: true

# Detalhe da região do olho
y0, y1, x0, x1 = 580, 740, 680, 900
crop = lambda img: img[y0:y1, x0:x1]

img_gray_crop = crop(img_gray)

sizes = [3, 7, 15]
imgs  = [img_gray_crop] + \
    [mm.blur(img_gray_crop, k) for k in sizes] # ou
    #[mm.conv(img_gray_crop, np.ones((k,k), dtype=np.float32)/(k*k)) for k in sizes]
titles = ["Original"] + [f"Média {k}×{k}" for k in sizes]

mm.show(imgs, titles=titles, cols=4)

### Filtro Gaussiano

O filtro Gaussiano pesa os pixels da vizinhança de acordo com uma função Gaussiana bidimensional:

$$
G(s,t) = \frac{1}{2\pi\sigma^2}\,e^{-\frac{s^2+t^2}{2\sigma^2}}
$$ {#eq-03-gaussiana}

onde $\sigma$ é o desvio padrão e controla o raio de influência. Pixels mais próximos do centro têm peso maior; pixels distantes são progressivamente ignorados.

A @fig-03-gauss-kernel apresenta o *kernel* Gaussiano $5\times5$ gerado para $\sigma=1$. O *kernel* foi construído a partir do produto externo de dois vetores Gaussianos unidimensionais e posteriormente normalizado para que a soma de seus coeficientes seja igual a $1$. Observa-se que os maiores pesos concentram-se no centro da matriz, decrescendo radialmente em direção às bordas. Essa distribuição faz com que os pixels centrais tenham maior influência no resultado da filtragem, contribuindo para uma suavização mais natural e com melhor preservação de bordas do que o filtro de média.

In [ ]:
#[py]#
#| label: fig-03-gauss-kernel
#| fig-cap: "*Kernel* Gaussiano 5×5 (σ=1): pesos normalizados, maiores no centro e decrescentes radialmente. A grade facilita a leitura de cada coeficiente."
#| echo: true
#| output: true

# Gera kernel Gaussiano 5×5 via cv2
k_gauss = cv2.getGaussianKernel(5, 1)
w_gauss = k_gauss @ k_gauss.T          # @ => multiplicação matricial: G(s,t) = G(s)·G(t)
w_gauss = w_gauss / w_gauss.sum()      # garante soma = 1

print("Kernel Gaussiano 5×5 (σ=1), normalizado:")
for row in w_gauss:
    print("  " + "  ".join(f"{v:.4f}" for v in row))
print(f"\nSoma dos coeficientes: {w_gauss.sum():.6f}")
print(f"Peso central vs. canto: {w_gauss[2,2]:.4f} vs. {w_gauss[0,0]:.4f} "
      f"({w_gauss[2,2]/w_gauss[0,0]:.1f}× maior)")

mm.drawImgPlt((w_gauss * 1000).astype(np.uint8), scale=40)

In [ ]:
#[cpp]#
#| label: fig-03-gauss-kernel
#| fig-cap: "*Kernel* Gaussiano 5×5 (σ=1): resposta ao impulso do filtro — pesos maiores no centro, decrescendo radialmente."
#| echo: true
#| output: true

w = mm.gaussKernel(5, 1)   # mm::Kernel::gaussian(5, 1.0) na morph.hpp

print("Kernel Gaussiano 5x5 (s=1), normalizado:")
for y in range(5):
    print("  " + "  ".join(f"{w[y, x]:.4f}" for x in range(5)))
print(f"Peso central [2,2] = {w[2, 2]:.4f}   |   canto [0,0] = {w[0, 0]:.4f}")

# Visualização: resposta do filtro Gaussiano a um impulso central
impulso = np.zeros((5, 5), dtype=np.uint8)
impulso[2, 2] = 255
mm.drawImgPlt(mm.gaussian(impulso, 5, 1), scale=40)

::: {.callout-note}

### Vantagem computacional da separabilidade {.unnumbered}

Considere um *kernel* quadrado de tamanho $n \times n$. Se esse filtro for separável (como o Gaussiano), a convolução 2D pode ser decomposta em duas convoluções 1D: uma horizontal e outra vertical.

Nesse caso, o custo por pixel passa de aproximadamente $O(n^2)$ operações (convolução 2D direta) para $O(2n)$ operações (duas convoluções 1D). Assim, a complexidade é reduzida de forma significativa, tornando o processamento mais eficiente

:::

Comparado ao filtro de média, o Gaussiano:

- **Preserva melhor as bordas** — a ponderação radial suaviza sem criar transições abruptas;
- **Não introduz anéis** (*ringing*) no domínio da frequência, pois a Gaussiana é sua própria transformada de Fourier (Capítulo 5);
- **É controlado por $\sigma$** — aumentar $\sigma$ equivale a aumentar o raio de suavização de forma contínua e previsível.

A @fig-03-gauss compara os filtros de média e Gaussiano aplicados à imagem do leopardo usando uma janela $9\times9$. O filtro de média foi implementado por convolução com um *kernel* uniforme, em que todos os $81$ pixels da vizinhança possuem o mesmo peso ($1/81$), enquanto o filtro Gaussiano foi obtido com `cv2.GaussianBlur`, utilizando pesos definidos por uma distribuição Gaussiana. Ambos reduzem ruído e suavizam a imagem, porém o filtro Gaussiano preserva melhor as bordas e os detalhes locais, como pode ser observado na região ampliada do olho.

A @fig-03-gauss compara os filtros de média e Gaussiano aplicados a um detalhe da imagem do leopardo com *kernels* $9\times 9$. O filtro de média foi obtido com `mm.blur`, equivalente à convolução com um *kernel* uniforme cujos coeficientes valem $1/81$, enquanto o filtro Gaussiano foi obtido com `mm.gaussian`, equivalente à convolução com um *kernel* gerado a partir de uma distribuição Gaussiana. Ambos promovem suavização e redução de ruído, porém o filtro Gaussiano atribui maior peso aos pixels centrais da vizinhança, preservando melhor as bordas e os detalhes locais, como pode ser observado na região ampliada do olho.


In [ ]:
#| label: fig-03-gauss
#| fig-cap: "Comparação entre filtro de média e Gaussiano (*kernel* 9×9, σ=0). O Gaussiano preserva melhor as bordas, visível no detalhe do rosto."
#| echo: true
#| output: true

img_media9 = mm.blur(img_gray, 9)
img_gauss9 = mm.gaussian(img_gray, 9, 0)

# Detalhe da região do olho
img_gray_crop  = mm.crop(img_gray,   580, 740, 680, 900)
img_media9_crop = mm.crop(img_media9, 580, 740, 680, 900)
img_gauss9_crop = mm.crop(img_gauss9, 580, 740, 680, 900)

mm.show(
    [img_gray_crop, img_media9_crop, img_gauss9_crop],
    titles=["Detalhe: Original", "Média 9×9", "Gaussiano 9×9"],
    cols=3, figsize=(12, 8)
)

## Filtragem Espacial de Realce

Os filtros de realce (*sharpening filters*) enfatizam transições abruptas de intensidade, aumentando a nitidez e a visibilidade de bordas. São filtros **passa-alta** — amplificam as componentes de alta frequência (bordas, textura) e suprimem as de baixa frequência (regiões uniformes).

A intuição é simples: se subtrairmos de uma imagem sua versão suavizada (que contém apenas as baixas frequências), o que resta são as altas frequências — bordas e detalhes. Somando esse resíduo de volta à imagem original, o contraste local aumenta:

$$
g = f + k\,(f - f_{\text{suave}}), \quad k > 0
$$ {#eq-03-realce-intuitivo}

O @fig-03-sim-03-filtragem1d ilustra esse processo em um sinal 1D sintético com três estruturas distintas: um degrau largo, um pico fino e uma rampa suave. No painel ①, o sinal original $f(x)$; no ②, a versão suavizada $f_{\text{suave}}(x)$ obtida por média móvel — note como o pico fino é atenuado. O painel ③ exibe o resíduo $f - f_{\text{suave}}$, que retém apenas as transições abruptas. Por fim, o painel ④ mostra $g(x)$: o pico, antes atenuado, é restaurado e amplificado em relação ao original. Ajuste $k$ e o tamanho da janela para observar o *trade-off* entre nitidez e amplificação de ruído.

Os filtros de realce formalizam essa ideia diretamente no *kernel*, sem precisar de duas etapas separadas.

In [ ]:
#| label: fig-03-sim-03-filtragem1d
#| fig-cap: "Simulador: Filtragem Espacial de Realce 1D (Unsharp Masking e High-Boost)"
#| echo: false
#| output: true

from IPython.display import HTML
HTML("""
<div id="sim-03-filtragem1d" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🎮 Simulador: Filtragem Espacial de Realce 1D</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = f + k·(f − f_suave)</span>
  </div>

  <div style="padding:16px;background:#ffffff;">

    <!-- Controles -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;display:flex;flex-wrap:wrap;gap:16px;align-items:flex-end;">
      
      <div style="display:flex;flex-direction:column;gap:4px;flex:1;min-width:140px;">
        <label style="font-size:11px;font-weight:700;color:#b9770e;">Parâmetro k (Realce)</label>
        <input id="sim_cap03_k_slider" type="range" min="0" max="5" step="0.1" value="1.5" style="width:100%;cursor:pointer;">
        <span style="font-size:11px;font-family:monospace;font-weight:700;color:#26241d;">k = <span id="sim_cap03_k_val">1.5</span></span>
      </div>

      <div style="display:flex;flex-direction:column;gap:4px;flex:1;min-width:140px;">
        <label style="font-size:11px;font-weight:700;color:#2980b9;">Janela de Suavização (pts)</label>
        <input id="sim_cap03_win_slider" type="range" min="3" max="31" step="2" value="9" style="width:100%;cursor:pointer;">
        <span style="font-size:11px;font-family:monospace;font-weight:700;color:#26241d;">janela = <span id="sim_cap03_win_val">9</span></span>
      </div>

      <div style="display:flex;flex-direction:column;gap:4px;flex:1;min-width:140px;">
        <label style="font-size:11px;font-weight:700;color:#8e44ad;">Nível de Ruído σ</label>
        <input id="sim_cap03_noise_slider" type="range" min="0" max="0.3" step="0.01" value="0.04" style="width:100%;cursor:pointer;">
        <span style="font-size:11px;font-family:monospace;font-weight:700;color:#26241d;">σ = <span id="sim_cap03_noise_val">0.04</span></span>
      </div>

    </div>

    <!-- Gráficos com Fundos Pastéis -->
    <div style="display:flex;flex-direction:column;gap:6px;">

      <!-- ① Azul Pastéis -->
      <div style="background:#ebf4fd;border:1px solid #d4e5f7;border-radius:8px;padding:8px 10px;">
        <div style="font-size:11px;font-weight:700;color:#1a5fa8;margin-bottom:3px;font-family:monospace;">① f(x) — Sinal Original</div>
        <canvas id="sim_cap03_c1" height="80" style="width:100%;display:block;"></canvas>
      </div>

      <div style="text-align:center;font-size:14px;color:#8a8371;line-height:1;">↓ filtro passa-baixa (média móvel)</div>

      <!-- ② Verde Pastéis -->
      <div style="background:#eaf7f2;border:1px solid #d1efe3;border-radius:8px;padding:8px 10px;">
        <div style="font-size:11px;font-weight:700;color:#0d6b4f;margin-bottom:3px;font-family:monospace;">② f_suave(x) — Pico Atenuado pelo Filtro</div>
        <canvas id="sim_cap03_c2" height="80" style="width:100%;display:block;"></canvas>
      </div>

      <div style="text-align:center;font-size:14px;color:#8a8371;line-height:1;">↓ subtração: f − f_suave</div>

      <!-- ③ Salmão/Rosa Pastéis -->
      <div style="background:#fef0eb;border:1px solid #fcdad0;border-radius:8px;padding:8px 10px;">
        <div style="font-size:11px;font-weight:700;color:#a03010;margin-bottom:3px;font-family:monospace;">③ Resíduo (f − f_suave) — Altas Frequências / Bordas</div>
        <canvas id="sim_cap03_c3" height="80" style="width:100%;display:block;"></canvas>
      </div>

      <div style="text-align:center;font-size:14px;color:#8a8371;line-height:1;">↓ soma: f + k · resíduo</div>

      <!-- ④ Roxo Pastéis -->
      <div style="background:#f2f0fd;border:1px solid #e1dcf9;border-radius:8px;padding:8px 10px;">
        <div style="font-size:11px;font-weight:700;color:#4a3faa;margin-bottom:3px;font-family:monospace;">④ g(x) — Sinal com Pico Realçado</div>
        <canvas id="sim_cap03_c4" height="80" style="width:100%;display:block;"></canvas>
      </div>

    </div>

  </div>
</div>

<script>
(function(){
  function initSimCap03(root){
    if (!root || root.dataset.simCap03Init) return;
    root.dataset.simCap03Init = "1";

    var N = 250;
    var peak_i = 115;

    var slK     = root.querySelector('#sim_cap03_k_slider');
    var slWin   = root.querySelector('#sim_cap03_win_slider');
    var slNoise = root.querySelector('#sim_cap03_noise_slider');

    var valK     = root.querySelector('#sim_cap03_k_val');
    var valWin   = root.querySelector('#sim_cap03_win_val');
    var valNoise = root.querySelector('#sim_cap03_noise_val');

    function make_signal(noise) {
      var seed = 42;
      function rand() { seed = (seed * 9301 + 49297) % 233280; return seed / 233280; }
      function randn() { return Math.sqrt(-2 * Math.log(rand() + 1e-9)) * Math.cos(2 * Math.PI * rand()); }
      var f = [];
      for (var i = 0; i < N; i++) {
        var v = 0.2;
        if (i >= 30  && i <= 80)  v += 0.7;
        if (i >= 110 && i <= 120) v += 1.0;
        if (i >= 150 && i <= 190) v += 0.5 * (i - 150) / 40;
        v += noise * randn();
        f.push(v);
      }
      return f;
    }

    function moving_avg(f, win) {
      var half = Math.floor(win / 2);
      return f.map(function(_, i){
        var s = 0, c = 0;
        for (var j = Math.max(0, i - half); j <= Math.min(f.length - 1, i + half); j++) {
          s += f[j];
          c++;
        }
        return s / c;
      });
    }

    function drawCanvas(canvasId, datasets, annotations) {
      var canvas = root.querySelector('#' + canvasId);
      if (!canvas) return;
      canvas.width = canvas.offsetWidth || 800;
      var ctx = canvas.getContext('2d');
      var W = canvas.width, H = canvas.height;
      ctx.clearRect(0, 0, W, H);

      var padL = 8, padR = 8, padT = 6, padB = 6;
      var allVals = [].concat.apply([], datasets.map(function(d){ return d.data; }));
      var mn = Math.min.apply(null, allVals);
      var mx = Math.max.apply(null, allVals);
      var rng = mx - mn || 1;
      var drawH = H - padT - padB, drawW = W - padL - padR;

      function toY(v){ return H - padB - (v - mn) / rng * drawH; }
      function toX(i){ return padL + i / (N - 1) * drawW; }

      // Faixa de destaque do pico
      var xA = toX(108), xB = toX(122);
      ctx.fillStyle = 'rgba(255, 200, 50, 0.15)';
      ctx.fillRect(xA, padT, xB - xA, drawH);

      // Grade
      ctx.strokeStyle = 'rgba(38, 36, 29, 0.08)';
      ctx.lineWidth = 0.5;
      for (var g = 0; g <= 3; g++) {
        var gy = padT + g / 3 * drawH;
        ctx.beginPath(); ctx.moveTo(padL, gy); ctx.lineTo(W - padR, gy); ctx.stroke();
      }

      // Linha do zero
      if (mn < 0 && mx > 0) {
        ctx.strokeStyle = 'rgba(38, 36, 29, 0.2)';
        ctx.lineWidth = 0.8;
        ctx.setLineDash([4, 3]);
        var y0 = toY(0);
        ctx.beginPath(); ctx.moveTo(padL, y0); ctx.lineTo(W - padR, y0); ctx.stroke();
        ctx.setLineDash([]);
      }

      // Desenho das séries
      datasets.forEach(function(ds){
        ctx.strokeStyle = ds.color;
        ctx.lineWidth   = ds.width || 1.8;
        ctx.globalAlpha = ds.alpha || 1;
        if (ds.dash) ctx.setLineDash(ds.dash); else ctx.setLineDash([]);
        ctx.beginPath();
        ds.data.forEach(function(v, i){
          if (i === 0) ctx.moveTo(toX(i), toY(v)); else ctx.lineTo(toX(i), toY(v));
        });
        ctx.stroke();
        ctx.setLineDash([]);
        ctx.globalAlpha = 1;
      });

      // Anotações
      if (annotations) {
        annotations.forEach(function(an){
          var xi = toX(an.i), yi = toY(an.v);
          ctx.strokeStyle = an.color || '#5e5a4a';
          ctx.lineWidth = 1;
          ctx.setLineDash([3, 3]);
          ctx.beginPath(); ctx.moveTo(xi, padT); ctx.lineTo(xi, H - padB); ctx.stroke();
          ctx.setLineDash([]);

          ctx.fillStyle = an.color || '#5e5a4a';
          ctx.beginPath(); ctx.arc(xi, yi, 4, 0, 2 * Math.PI); ctx.fill();

          ctx.fillStyle = an.color || '#26241d';
          ctx.font = 'bold 10px monospace';
          ctx.fillText(an.label, xi + 6, Math.max(padT + 12, Math.min(H - padB - 4, yi - 6)));
        });
      }
    }

    function update() {
      var k     = parseFloat(slK.value) || 0;
      var win   = parseInt(slWin.value) || 3;
      var noise = parseFloat(slNoise.value) || 0;

      valK.textContent     = k.toFixed(1);
      valWin.textContent   = win;
      valNoise.textContent = noise.toFixed(2);

      var f       = make_signal(noise);
      var f_suave = moving_avg(f, win);
      var residuo = f.map(function(v, i){ return v - f_suave[i]; });
      var g       = f.map(function(v, i){ return v + k * residuo[i]; });

      var pk = peak_i;
      var peakF = f[pk], peakS = f_suave[pk], peakR = residuo[pk], peakG = g[pk];

      // ① Original (Azul)
      drawCanvas('sim_cap03_c1',
        [{ data: f, color: '#1a5fa8' }],
        [{ i: pk, v: peakF, color: '#1a5fa8', label: 'pico' }]
      );

      // ② Suavizado (Verde)
      drawCanvas('sim_cap03_c2',
        [{ data: f, color: '#1a5fa8', width: 1, alpha: 0.35, dash: [4, 3] },
         { data: f_suave, color: '#0d6b4f', width: 2 }],
        [{ i: pk, v: peakS, color: '#0d6b4f', label: 'atenuado' }]
      );

      // ③ Resíduo (Salmão / Laranja)
      drawCanvas('sim_cap03_c3',
        [{ data: residuo, color: '#a03010' }],
        [{ i: pk, v: peakR, color: '#a03010', label: 'borda detectada' }]
      );

      // ④ Realçado (Roxo)
      drawCanvas('sim_cap03_c4',
        [{ data: f, color: '#1a5fa8', width: 1, alpha: 0.35, dash: [4, 3] },
         { data: g, color: '#4a3faa', width: 2.2 }],
        [{ i: pk, v: peakG, color: '#4a3faa', label: 'pico realçado' }]
      );
    }

    [slK, slWin, slNoise].forEach(function(sl){
      sl.addEventListener('input', update);
    });

    update();
  }

  function tryInitSimCap03(){
    var root = document.getElementById('sim-03-filtragem1d');
    if (root) initSimCap03(root); else setTimeout(tryInitSimCap03, 200);
  }
  tryInitSimCap03();
})();
</script>
</div>
""")

### Laplaciano

O Laplaciano é um operador de **segunda derivada** isotrópico, ou seja, responde igualmente a variações em todas as direções, ao contrário de operadores de primeira derivada, como Sobel e Prewitt, que são direcionais:

$$
\nabla^2 f = \frac{\partial^2 f}{\partial x^2} + \frac{\partial^2 f}{\partial y^2}
$$ {#eq-03-laplaciano}

Uma propriedade importante da segunda derivada é que seu valor é **próximo de zero em regiões uniformes** e elevado nas transições de intensidade. Assim, ao subtrair o Laplaciano da imagem original, reforçam-se bordas e detalhes, aumentando o contraste local:

$$
g(x,y) = f(x,y) - \nabla^2 f(x,y)
$$ {#eq-03-realce-lap}

Na forma discreta, a segunda derivada em $x$ é aproximada por $f(x+1,y) - 2f(x,y) + f(x-1,y)$, e analogamente em $y$. Somando as duas direções, obtém-se o *kernel* $w_4$ (4-vizinhos) ou $w_8$ (8-vizinhos, incluindo diagonais):

$$
w_4 = \begin{bmatrix} 0 & 1 & 0 \\ 1 & -4 & 1 \\ 0 & 1 & 0 \end{bmatrix}, \qquad
w_8 = \begin{bmatrix} 1 & 1 & 1 \\ 1 & -8 & 1 \\ 1 & 1 & 1 \end{bmatrix}
$$ {#eq-03-laplaciano-kernel}

::: {.callout-note}
### Soma zero e centro negativo {.unnumbered}
Ambos os *kernels* têm **soma dos coeficientes igual a zero**: em regiões uniformes, a saída é 0 — o Laplaciano não altera o brilho médio, apenas detecta variações. O **centro negativo** indica que o pixel é comparado com seus vizinhos: quanto mais ele se destacar (para cima ou para baixo), maior o valor absoluto do Laplaciano naquele ponto.
:::

No exemplo a seguir, o pixel central $[1,1]=107$ possui vizinhos $\{95, 106, 108, 103\}$. Como esses valores são próximos entre si, a região é quase uniforme e o Laplaciano retorna um valor baixo, produzindo pouco realce. Em regiões de borda, onde há diferenças maiores entre o pixel central e seus vizinhos, o Laplaciano assume valores mais elevados (positivos ou negativos), e a operação de @eq-03-realce-lap intensifica essas transições.

A @fig-03-laplaciano-patch ilustra o cálculo do Laplaciano com o *kernel* $w_4$ em uma vizinhança $3\times3$ destacada dentro de um *patch* $5\times5$. O exemplo mostra o valor obtido pelo operador e o correspondente pixel realçado na imagem de saída, que passa de 107 para 123.

In [ ]:
#[py]#
#| label: fig-03-laplaciano-patch
#| fig-cap: "*Kernel* Laplaciano w4 aplicado ao *patch* 5×5: a janela amarela destaca a vizinhança 3×3 onde o operador de segunda derivada é calculado."
#| echo: true
#| output: true

patch  = img_gray[250:255, 250:255]
w4     = np.array([[0, 1, 0],
                   [1,-4, 1],
                   [0, 1, 0]], dtype=np.float32)
B      = np.ones((3,3), dtype=np.uint8)

p      = patch.astype(np.float32)
lap_11 = int(p[0,1] + p[1,0] - 4*p[1,1] + p[1,2] + p[2,1])
real_11 = int(np.clip(p[1,1] - lap_11, 0, 255))

print("Patch original:")
print(mm.drawImg(patch))
print(f"Laplac. w4 em  [1,1]: ", end="")
print(f"{p[0,1]:.0f} + {p[1,0]:.0f} - 4×{p[1,1]:.0f} + {p[1,2]:.0f} + {p[2,1]:.0f} = {lap_11}")
print(f"Pixel realçado [1,1]: {p[1,1]:.0f} - ({lap_11}) = {real_11}")

mm.drawImgKernel(patch, B, x=1, y=1, scale=40.0)

In [ ]:
#[cpp]#
#| label: fig-03-laplaciano-patch
#| fig-cap: "*Kernel* Laplaciano w4 sobre o *patch* 5×5: a janela amarela destaca a vizinhança 3×3 onde o operador de segunda derivada é calculado."
#| echo: true
#| output: true

patch = mm.crop(img_gray, 250, 255, 250, 255)
B     = np.ones((3, 3), dtype=np.uint8)

print("Patch 5x5 (intensidades):")
print(mm.drawImg(patch))

mm.drawImgKernel(patch, B, 1, 1, 40)

A @fig-03-laplaciano compara a aplicação dos *kernels* Laplacianos $w_4$ e $w_8$ em um recorte maior da imagem do leopardo. Para cada caso, são mostradas a resposta bruta do operador, que evidencia as bordas e transições de intensidade, e a imagem obtida após o realce por subtração do Laplaciano. Observa-se que o *kernel* $w_8$, por considerar também os vizinhos diagonais, produz uma resposta mais intensa e detecta variações em mais direções, resultando em um realce ligeiramente mais acentuado.

In [ ]:
#| label: fig-03-laplaciano
#| fig-cap: "Laplaciano aplicado à imagem do leopardo: resposta bruta (bordas) com w4 e w8, e imagens realçadas pela subtração do Laplaciano. w8 é mais sensível às diagonais."
#| echo: true
#| output: true

w4 = np.array([[0, 1, 0],
               [1,-4, 1],
               [0, 1, 0]], dtype=np.float32)
w8 = np.array([[1, 1, 1],
               [1,-8, 1],
               [1, 1, 1]], dtype=np.float32)

mm.show(
    [img_gray_crop, mm.laplacian_viz(img_gray_crop, w4), mm.laplacian(img_gray_crop, w4),
     img_gray_crop, mm.laplacian_viz(img_gray_crop, w8), mm.laplacian(img_gray_crop, w8)],
    titles=["Original", "Laplaciano w4", "Realce w4",
            "Original", "Laplaciano w8", "Realce w8"],
    cols=3, figsize=(14, 8)
)

### Operador de Sobel

O operador de Sobel estima as **derivadas parciais de primeira ordem** nas direções horizontal e vertical. Diferente do Laplaciano (segunda derivada), o Sobel é direcional e mais robusto ao ruído, pois cada *kernel* combina uma derivada com uma suavização Gaussiana perpendicular:

$$
G_x = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix} * f, \qquad
G_y = \begin{bmatrix} -1 & -2 & -1 \\ 0 & 0 & 0 \\ 1 & 2 & 1 \end{bmatrix} * f
$$ {#eq-03-sobel}

$G_x$ detecta bordas **verticais** (variação na direção $x$); $G_y$ detecta bordas **horizontais** (variação na direção $y$). Os pesos $\{1,2,1\}$ na direção perpendicular correspondem à suavização Gaussiana 1D, que reduz a sensibilidade ao ruído.

::: {.callout-note}
### Sobel é correlação, não convolução {.unnumbered}
Os *kernels* de Sobel são **assimétricos** — a rotação de 180° altera o resultado. `cv2.Sobel` implementa correlação cruzada (como `cv2.filter2D`). Para obter a derivada direcional correta, os sinais já estão definidos para correlação: $G_x$ retorna valores positivos onde a intensidade cresce da esquerda para a direita.
:::

A magnitude do **gradiente** combina os dois componentes, representando a força da borda independente de direção:

$$
|\nabla f| = \sqrt{G_x^2 + G_y^2}
$$ {#eq-03-gradiente}

E a **direção** do gradiente (perpendicular à borda) é:

$$
\theta = \arctan\left(\frac{G_y}{G_x}\right)
$$ {#eq-03-direcao}

Para ilustrar numericamente, calcula-se $G_x$ e $G_y$ manualmente no pixel central $[1,1]$ do *patch* 5×5:

In [ ]:
#[py]#
#| echo: true
#| output: true

patch = img_gray[250:255, 250:255]
p     = patch.astype(np.float32)

# Gx no pixel [1,1]: coluna direita - coluna esquerda (ponderada)
gx = (p[0,2] + 2*p[1,2] + p[2,2]) - (p[0,0] + 2*p[1,0] + p[2,0])

# Gy no pixel [1,1]: linha inferior - linha superior (ponderada)
gy = (p[2,0] + 2*p[2,1] + p[2,2]) - (p[0,0] + 2*p[0,1] + p[0,2])

mag   = np.sqrt(gx**2 + gy**2)
theta = np.degrees(np.arctan2(gy, gx))

print("Patch 5x5:")
print(mm.drawImg(patch))
print(f"Gx em [1,1]: ({p[0,2]:.0f} + 2*{p[1,2]:.0f} + {p[2,2]:.0f}) \
      - ({p[0,0]:.0f} + 2*{p[1,0]:.0f} + {p[2,0]:.0f}) = {gx:.1f}")
print(f"Gy em [1,1]: ({p[2,0]:.0f} + 2*{p[2,1]:.0f} + {p[2,2]:.0f}) \
      - ({p[0,0]:.0f} + 2*{p[0,1]:.0f} + {p[0,2]:.0f}) = {gy:.1f}")

# Substituído o | pelo prefixo mag para não quebrar o interpretador de tabelas do Quarto
print(f"mag(Grad f)  = sqrt({gx:.1f}^2 + {gy:.1f}^2) = {mag:.1f}")
print(f"Theta        = arctan({gy:.1f}/{gx:.1f}) = {theta:.1f} graus")

O valor reduzido de $|{\nabla f}|$ nesse *patch* confirma que a região é quase uniforme, pois o gradiente assume valores elevados apenas onde há mudanças significativas de intensidade. A @fig-03-sobel aplica o operador de Sobel a um recorte maior da imagem do leopardo. São apresentadas as respostas horizontal ($G_x$) e vertical ($G_y$), obtidas por convolução com os respectivos *kernels* de Sobel, além da magnitude $|{\nabla f}|$, calculada a partir da combinação de ambas. Enquanto $G_x$ destaca bordas verticais e $G_y$ bordas horizontais, a magnitude evidencia bordas em qualquer direção.

In [ ]:
#[py]#
#| label: fig-03-sobel
#| fig-cap: "Operador de Sobel na imagem do leopardo: Gx detecta bordas verticais, Gy detecta bordas horizontais, e a magnitude |∇f| combina ambos, revelando todas as bordas independente de direção."
#| echo: true
#| output: true

def norm255(img):
    mn, mx = img.min(), img.max() #+1e-9 para evitar divisão por zero
    return ((img - mn) / (mx - mn + 1e-9) * 255).astype(np.uint8)

sobelx    = cv2.Sobel(img_gray_crop, cv2.CV_64F, 1, 0, ksize=3)
sobely    = cv2.Sobel(img_gray_crop, cv2.CV_64F, 0, 1, ksize=3)
magnitude = np.sqrt(sobelx**2 + sobely**2)
direcao   = np.degrees(np.arctan2(sobely, sobelx))

# magnitude = mm.sobel(img_gray_crop) # ou, mas com clip em vez de norm255

mm.show(
    [img_gray_crop, norm255(np.abs(sobelx)), norm255(np.abs(sobely)),
     norm255(magnitude), norm255(direcao % 180)],
    titles=["Original", "Gx (bordas vert.)", "Gy (bordas horiz.)",
            "Magnitude |∇f|", "Direção θ (mod 180°)"],
    cols=5
)

In [ ]:
#[cpp]#
#| label: fig-03-sobel
#| fig-cap: "Operador de Sobel na imagem do leopardo: a magnitude |∇f| combina os gradientes horizontal e vertical, revelando todas as bordas. A decomposição Gx/Gy com sinal fica na trilha Python — mm::sobel devolve a magnitude já com clip."
#| echo: true
#| output: true

mag = mm.sobel(img_gray_crop)

mm.show([img_gray_crop, mag],
        titles=["Original", "Magnitude |grad f| (mm.sobel)"], cols=2)

### Operador de Prewitt

O operador de Prewitt é estruturalmente idêntico ao Sobel, mas substitui a ponderação Gaussiana $\{1,2,1\}$ por pesos uniformes $\{1,1,1\}$:

$$
G_x = \begin{bmatrix} -1 & 0 & 1 \\ -1 & 0 & 1 \\ -1 & 0 & 1 \end{bmatrix} * f, \qquad
G_y = \begin{bmatrix} -1 & -1 & -1 \\ 0 & 0 & 0 \\ 1 & 1 & 1 \end{bmatrix} * f
$$ {#eq-03-prewitt}

A magnitude e a direção do gradiente seguem as mesmas equações do Sobel (@eq-03-gradiente e @eq-03-direcao). A diferença prática é que o Prewitt é ligeiramente mais sensível ao ruído — a suavização perpendicular uniforme pondera menos o pixel central da linha — mas computacionalmente mais simples. Em imagens com baixo ruído os resultados são equivalentes.

In [ ]:
#[py]#
#| label: fig-03-prewitt
#| fig-cap: "Operador de Prewitt: Gx, Gy e magnitude — comparável ao Sobel, mas sem ponderação Gaussiana perpendicular."
#| echo: true
#| output: true

Kx = np.array([[-1,0,1],[-1,0,1],[-1,0,1]], dtype=np.float32)
Ky = np.array([[-1,-1,-1],[0,0,0],[1,1,1]], dtype=np.float32)
f  = img_gray_crop.astype(np.float32)

mm.show(
    [img_gray_crop,
     norm255(np.abs(cv2.filter2D(f, -1, Kx))),
     norm255(np.abs(cv2.filter2D(f, -1, Ky))),
     mm.prewitt(img_gray_crop)],
    titles=["Original", "Gx", "Gy", "Magnitude |∇f|"],
    cols=4
)

In [ ]:
#[cpp]#
#| label: fig-03-prewitt
#| fig-cap: "Operador de Prewitt: magnitude do gradiente, comparável ao Sobel mas sem a ponderação central. mm::prewitt devolve |∇f| com clip."
#| echo: true
#| output: true

mm.show([img_gray_crop, mm.prewitt(img_gray_crop)],
        titles=["Original", "Magnitude |grad f| (mm.prewitt)"], cols=2)

### *Unsharp Masking* (USM)

O *Unsharp Masking* é uma técnica clássica de realce de nitidez originária da fotografia analógica, hoje amplamente usada em software de edição de imagens. A ideia central é extrair as **componentes de alta frequência** da imagem (bordas e detalhes) e somá-las de volta à original com um peso $k$:

| Etapa | Operação | Descrição |
|:-----:|:---------|:----------|
| 1 | $\bar{f} = f * G_\sigma$ | Suaviza com Gaussiana — retém baixas frequências |
| 2 | $m = f - \bar{f}$ | Máscara: diferença = altas frequências (bordas) |
| 3 | $g = f + k \cdot m$ | Soma ponderada da máscara à original |

: Etapas do *Unsharp Masking*. {#tbl-03-usm}

Substituindo a etapa 2 na etapa 3, obtém-se a expressão compacta:

$$
g = f + k\,(f - f*G_\sigma) = (1+k)\,f - k\,(f*G_\sigma)
$$ {#eq-03-usm}

O parâmetro $k$ controla a intensidade do realce:

- $k = 0$: sem realce ($g = f$);
- $k = 1$: USM clássico — duplica a contribuição das altas frequências;
- $k > 1$: *High Boost Filtering* — amplificação além do dobro, útil para imagens muito borradas.

::: {.callout-warning}
### Amplificação de ruído {.unnumbered}
O USM não distingue bordas de ruído — ambos são componentes de alta frequência. Para $k$ elevado, o ruído presente na imagem é amplificado junto com as bordas. Por isso, é recomendável aplicar uma leve suavização antes do USM em imagens ruidosas, ou usar $\sigma$ pequeno na Gaussiana.
:::

Para ilustrar as etapas do USM, a @fig-03-usm2 aplica o método a um *patch* $30\times30$ da imagem do leopardo, utilizando $\sigma=1$ e $k=1$. Inicialmente, a imagem é suavizada por um filtro Gaussiano. Em seguida, a máscara de alta frequência é obtida pela diferença entre a imagem original e a suavizada. Por fim, essa máscara é somada à imagem original, reforçando bordas e detalhes. A figura apresenta as três etapas do processo e o resultado final do realce.

In [ ]:
#[py]#
#| label: fig-03-usm2
#| fig-cap: "Realce de nitidez por *Unsharp Masking* na imagem do leopardo: a imagem suavizada é subtraída da original para gerar a máscara de alta frequência, que é então reintroduzida para realçar bordas e detalhes."
#| echo: true
#| output: true

patch  = img_gray_crop[35:65, 45:75]
k = 1.0
sigma = 1.0

p = patch.astype(np.float32)

# Etapa 1: suavização Gaussiana
ksize   = int(6 * sigma + 1) | 1  # operador binário garante tamanho ímpar
p_suave = mm.gaussian(patch, ksize, sigma).astype(np.float32)

# Etapa 2: máscara de alta frequência
mascara = p - p_suave

# Etapa 3: realce
p_usm = np.clip(p + k * mascara, 0, 255).astype(np.uint8)

# p_usm = mm.usm(patch, k)  # ou, direto

print(f"Pixel central [1,1]: original={int(p[1,1])} suave={p_suave[1,1]:.1f}"
      f" mascara={mascara[1,1]:.1f} realcado={p_usm[1,1]}")

mm.show(
    [patch,
     p_suave.astype(np.uint8),
     np.clip(mascara + 128, 0, 255).astype(np.uint8),  # negativos visíveis
     p_usm],
    titles=["Patch original",
            f"Suavizado (σ={sigma})",
            "Máscara (alta freq., +128)",
            f"Realçado USM (k={k})"],
    cols=4, figsize=(12, 4)
)

In [ ]:
#[cpp]#
#| label: fig-03-usm2
#| fig-cap: "Realce por *Unsharp Masking* num *patch* do leopardo: mm::usm faz suavização Gaussiana, subtrai da original (máscara de alta frequência) e reintroduz a máscara realçada."
#| echo: true
#| output: true

patch = mm.crop(img_gray_crop, 35, 65, 45, 75)

p_suave = mm.gaussian(patch, 7, 1.0)   # suavização Gaussiana
p_usm   = mm.usm(patch, 1.0)           # realce completo (k = 1.0)

mm.show([patch, p_suave, p_usm],
        titles=["Patch original", "Suavizado (σ=1)", "Realçado USM (k=1)"], cols=3)

A @fig-03-usm aplica o método USM a um recorte maior da imagem do leopardo utilizando $\sigma=1$ e diferentes valores do fator de ganho $k$. Em todos os casos, a máscara de alta frequência é obtida pela diferença entre a imagem original e sua versão suavizada por filtro Gaussiano. O parâmetro $k$ controla a intensidade do realce: valores menores produzem um aumento sutil de nitidez, enquanto valores maiores reforçam progressivamente bordas e detalhes. Observa-se que, para valores elevados de $k$, surgem halos ao redor das bordas e o ruído presente na imagem passa a ser amplificado.

In [ ]:
#| label: fig-03-usm
#| fig-cap: "*Unsharp Masking* na imagem do leopardo com σ=1 e k de 0.5 a 8.0. Para k>2 surgem halos nas bordas e o ruído de fundo aparece."
#| echo: true
#| output: true

mm.show(
    [img_gray_crop,
     mm.usm(img_gray_crop, 0.5), mm.usm(img_gray_crop, 1.0),
     mm.usm(img_gray_crop, 3.0), mm.usm(img_gray_crop, 5.0),
     mm.usm(img_gray_crop, 8.0)],
    titles=["Original", "USM k=0.5", "USM k=1.0", "USM k=3.0", "USM k=5.0", "USM k=8.0"],
    cols=3
)

### Detector de Canny

O Canny combina quatro etapas em sequência — suavização Gaussiana, gradiente de Sobel,
supressão de não-máximos e histerese por duplo limiar — para produzir bordas **finas,
binárias e conectadas**. Diferente de Sobel e Prewitt, o resultado não é um mapa de
gradiente contínuo, mas uma máscara onde cada pixel é borda ou não.

O parâmetro central é o par de limiares $(T_{low}, T_{high})$. Pixels com gradiente acima de
$T_{high}$ são bordas certas; abaixo de $T_{low}$, descartados. Os pixels ambíguos —
entre os dois limiares — são decididos por **histerese**: tornam-se borda se estiverem
conectados a uma borda certa, e descartados caso contrário. Isso evita tanto a perda de
trechos fracos de bordas reais quanto a inclusão de ruído isolado. Uma heurística comum
é $T_{high} = 3 \times T_{low}$.

In [ ]:
#| label: fig-03-canny
#| fig-cap: "Detector de Canny com diferentes pares de limiar: limiares baixos capturam mais bordas (inclusive ruído); limiares altos retêm apenas as bordas mais fortes."
#| echo: true
#| output: true

mm.show(
    [img_gray_crop,
     mm.canny(img_gray_crop, 30,  90),
     mm.canny(img_gray_crop, 60,  180),
     mm.canny(img_gray_crop, 120, 240)],
    titles=["Original", "Canny (30/90)", "Canny (60/180)", "Canny (120/240)"],
    cols=4
)


::: {.callout-note}
### Escolha dos limiares {.unnumbered}
Uma heurística comum é $T_{high} = 3 \times T_{low}$. Valores típicos dependem do intervalo de gradiente da imagem — `cv2.Canny` aceita valores absolutos em $[0, 255]$. Para imagens com contraste variável, calcular os limiares a partir de percentis da magnitude de Sobel é mais robusto do que valores fixos.
:::

## Filtros de Ordem: Filtro da Mediana

Os filtros de ordem (*order-statistic filters*) substituem o pixel central pelo valor de um **percentil** da distribuição de intensidades da vizinhança — ao contrário dos filtros lineares, que calculam combinações ponderadas. O mais importante é o **filtro da mediana**.

### Ruído Impulsivo: Sal e Pimenta

O ruído **sal e pimenta** (*salt-and-pepper noise*) substitui pixels aleatórios por valores extremos: 0 (pimenta, preto) ou 255 (sal, branco). É comum em transmissão de imagens com erros de bit e em câmeras com sensores defeituosos.

Para entender por que os filtros lineares falham, considere uma vizinhança 3×3 onde um único pixel foi corrompido para 255:

$$
\text{vizinhança} = \begin{bmatrix} 102 & 98 & 105 \\ 100 & \mathbf{255} & 97 \\ 103 & 99 & 101 \end{bmatrix}
$$

| Método | Cálculo | Resultado |
|:-------|:--------|----------:|
| Média | (102+98+...+255+...+101)/9 | ≈ 140 |
| Mediana | {97,98,99,100,**101**,102,103,105,255} | 101 |

: Média vs. mediana com um pixel corrompido. A mediana ignora o outlier; a média é deslocada ~40 níveis. {#tbl-03-mediana}

::: {.callout-warning}
### Por que filtros de média falham com ruído impulsivo? {.unnumbered}
A média é sensível a ***outliers*** — um único pixel com valor 255 em uma vizinhança de valor ≈ 100 eleva a saída para ≈ 140, espalhando o ruído pela imagem. A mediana, por ser um **estimador robusto**, seleciona o valor central da distribuição ordenada, descartando naturalmente os extremos sem nenhum ajuste especial.
:::

O exemplo a seguir ilustra o comportamento da média e da mediana na presença de um pixel corrompido por ruído impulsivo. Observa-se que a média é fortemente influenciada pelo valor extremo (255), produzindo uma estimativa distante dos valores predominantes da vizinhança. Já a mediana permanece próxima do valor original da região, evidenciando sua maior robustez a *outliers* e justificando seu uso na remoção de ruído sal e pimenta.

In [ ]:
#[py]#
#| echo: true
#| output: true

# Exemplo numérico: média vs. mediana com pixel corrompido
vizinhanca = np.array([102, 98, 105, 100, 255, 97, 103, 99, 101])
print(f"Vizinhança: {sorted([int(x) for x in vizinhanca])}")
print(f"Média:      {vizinhanca.mean():.1f}  (deslocada pelo outlier 255)")
print(f"Mediana:    {int(np.median(vizinhanca))}    (ignora o outlier)")
print(f"Valor original:  ~100")

A @fig-03-ruido apresenta o efeito do ruído sal e pimenta em diferentes densidades. O ruído foi gerado substituindo aleatoriamente uma fração dos pixels por valores mínimos (0, pimenta) e máximos (255, sal). À medida que a densidade aumenta de 2% para 10%, cresce a quantidade de pixels corrompidos, tornando a degradação visual mais evidente e dificultando a percepção de detalhes da imagem.

In [ ]:
#[py]#
#| label: fig-03-ruido
#| fig-cap: "Ruído sal e pimenta com densidades crescentes (2%, 5%, 10%). O parâmetro prob indica a fração total de pixels corrompidos, metade sal (255) e metade pimenta (0)."
#| echo: true
#| output: true

def add_salt_pepper(img, prob=0.05, seed=42):
    """Adiciona ruído sal e pimenta com probabilidade total prob."""
    noisy = img.copy()
    rnd   = np.random.default_rng(seed).random(img.shape)
    noisy[rnd < prob / 2]     = 0    # pimenta
    noisy[rnd > 1 - prob / 2] = 255  # sal
    n_corrompidos = int((rnd < prob/2).sum() + (rnd > 1-prob/2).sum())
    return noisy, n_corrompidos

probs = [0.02, 0.05, 0.10]
imgs_noise, titles_noise = [img_gray_crop], ["Original"]

for p in probs:
    noisy, n = add_salt_pepper(img_gray_crop, p)
    imgs_noise.append(noisy)
    titles_noise.append(f"Ruído {int(p*100)}%\n({n:,} pixels)")

mm.show(imgs_noise, titles=titles_noise, cols=4)

In [ ]:
#[cpp]#
#| label: fig-03-ruido
#| fig-cap: "Ruído sal e pimenta com densidades crescentes (2%, 5%, 10%): metade dos pixels corrompidos vira sal (255), metade pimenta (0)."
#| echo: true
#| output: true

def salt_pepper(img, prob):
    out = img.copy()
    h, w = out.shape[0], out.shape[1]
    rng = np.random.default_rng(42)
    n = int(prob * h * w)
    for _ in range(n):
        y = int(rng.integers(0, h))
        x = int(rng.integers(0, w))
        out[y, x] = 0 if rng.random() < 0.5 else 255
    return out

n2  = salt_pepper(img_gray_crop, 0.02)
n5  = salt_pepper(img_gray_crop, 0.05)
n10 = salt_pepper(img_gray_crop, 0.10)

mm.show([img_gray_crop, n2, n5, n10],
        titles=["Original", "Ruido 2%", "Ruido 5%", "Ruido 10%"], cols=4)

### Filtro da Mediana

O filtro da mediana substitui cada pixel pelo **valor mediano** dos pixels da sua vizinhança $n \times n$:

$$
g(x,y) = \text{med}_{(s,t) \in \mathcal{V}_{n}} \{f(x+s, y+t)\}
$$ {#eq-03-mediana}

O valor mediano é aquele que ocupa a posição central quando os $n^2$ valores da vizinhança são ordenados. Para uma janela $3\times3$ ($n^2=9$ pixels), a mediana é o 5º valor da sequência ordenada.

Para ilustrar, considere o mesmo *patch* 5×5 com um pixel corrompido artificialmente em $[1,1]$:

In [ ]:
#[py]#
#| echo: true
#| output: true

patch    = img_gray[250:255, 250:255].copy()
corrompido = patch.copy()
corrompido[1, 1] = 255   # injeta pixel sal no centro

# Vizinhança 3×3 centrada em [1,1]
viz_orig = sorted(patch[0:3, 0:3].ravel().tolist())
viz_corr = sorted(corrompido[0:3, 0:3].ravel().tolist())

print("Patch original:")
print(mm.drawImg(patch))
print("\nPatch com pixel corrompido [1,1]=255:")
print(mm.drawImg(corrompido))
print(f"\nVizinhança 3×3 original (ordenada):   {viz_orig}")
print(f"Mediana original:                       {viz_orig[4]}")
print(f"\nVizinhança 3×3 corrompida (ordenada): {viz_corr}")
print(f"Mediana corrompida:                     {viz_corr[4]}  ← ignora o 255")
print(f"Média  corrompida:                  {np.mean(viz_corr):.1f}  ← distorcida pelo 255")

O exemplo confirma: mesmo com o pixel corrompido a 255, a mediana retorna o valor central correto — o *outlier* ocupa a última posição na ordenação e é descartado naturalmente.

Por ser baseada em ordenação e não em soma, a mediana possui três propriedades fundamentais que a diferenciam dos filtros lineares:

- **Robusta** ao ruído impulsivo — outliers vão para as extremidades da sequência ordenada e não afetam o valor central;
- **Preservadora de bordas** — transições abruptas de intensidade são mantidas, pois a mediana seleciona um valor que já existe na vizinhança, sem criar novos níveis intermediários;
- **Não linear** — não pode ser expressa como convolução, portanto `mm.conv` não se aplica; usa-se `cv2.medianBlur`.

A @fig-03-ruido-filtros compara diferentes técnicas de remoção de ruído sal e pimenta aplicadas a uma imagem com 10% de pixels corrompidos. Foram avaliados os filtros Gaussiano, Média, Mediana, Bilateral e Morfológico (próximo capítulo), permitindo observar o compromisso entre remoção de ruído e preservação de detalhes. Em geral, os filtros de média e Gaussiano reduzem o ruído, mas tendem a borrar as bordas, enquanto a mediana apresenta melhor desempenho para ruído impulsivo. O filtro bilateral preserva melhor as bordas, e o filtro morfológico remove boa parte dos pixels corrompidos sem degradar excessivamente a estrutura da imagem.

In [ ]:
#[py]#
#| label: fig-03-ruido-filtros
#| fig-cap: "Comparação de filtros para remoção de ruído sal e pimenta (10%): Gaussiano, Média, Mediana, Bilateral e Morfológico. Linha superior: imagens completas; linha inferior: crop da região de interesse."
#| echo: true
#| output: true

# 10% de ruído para evidenciar diferenças entre filtros
noisy_5, _ = add_salt_pepper(img_gray_crop, prob=0.1) 

# ── Filtros ───────────────────────────────────────────────────────────────────
f_gauss   = cv2.GaussianBlur(noisy_5, (5, 5), 1.0)
f_media   = mm.conv(noisy_5, np.ones((5, 5), dtype=np.float32) / 20.0)
f_median3 = cv2.medianBlur(noisy_5, 3)
f_median5 = cv2.medianBlur(noisy_5, 5)
f_bilat   = cv2.bilateralFilter(noisy_5, d=9, sigmaColor=75, sigmaSpace=75)

# filtros morfológicos no próximo capítulo, mas já adiantados aqui para comparação
# B         = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
# f_morf    = cv2.morphologyEx(           # apresentado no próximo capítulo
#                 cv2.morphologyEx(noisy_5, cv2.MORPH_OPEN,  B),
#                 cv2.MORPH_CLOSE, B)
B = mm.secross()  # elemento estruturante de caixa 3×3
f_morf     = mm.asf(noisy_5, 'OC', B)  # equivalente via morph

# ── MSE ───────────────────────────────────────────────────────────────────────
def mse(a, b):
    return float(np.mean((a.astype(np.float32) - b.astype(np.float32))**2))

print(f"{'Filtro':<22} {'MSE':>8}")
print("-" * 32)
pares = [("Gaussiano 5×5",    f_gauss),
         ("Média 5×5",        f_media),
         ("Mediana 3×3",      f_median3),
         ("Mediana 5×5",      f_median5),
         ("Bilateral",        f_bilat),
         ("Morf. open+close", f_morf)]
for nome, img in pares:
    print(f"{nome:<22} {mse(img_gray_crop, img):>8.2f}")

imgs   = [img_gray_crop, noisy_5,    f_gauss,        f_media,
          f_median3,     f_median5,  f_bilat,        f_morf]
titles = ["Original",    "Ruído 10%", "Gaussiano 5×5","Média 5×5",
          "Mediana 3×3", "Mediana 5×5", "Bilateral", "Morf. open+close"]

mm.show(
    imgs,
    titles=[f"Crop: {t}" for t in titles],
    cols=4, rows=2, figsize=(14, 8)
)

In [ ]:
#[cpp]#
#| label: fig-03-ruido-filtros
#| fig-cap: "Filtros para ruído sal e pimenta (10%): Gaussiano, Média e Mediana. Bilateral e morfológico (open+close) ficam só na trilha Python — bilateral não tem equivalente em morph.hpp e morfologia é do próximo capítulo."
#| echo: true
#| output: true

def salt_pepper(img, prob):
    out = img.copy()
    h, w = out.shape[0], out.shape[1]
    rng = np.random.default_rng(42)
    for _ in range(int(prob * h * w)):
        y = int(rng.integers(0, h))
        x = int(rng.integers(0, w))
        out[y, x] = 0 if rng.random() < 0.5 else 255
    return out

noisy = salt_pepper(img_gray_crop, 0.10)

f_gauss   = mm.gaussian(noisy, 5, 1.0)
f_media   = mm.blur(noisy, 5)
f_median3 = mm.median(noisy, 3)
f_median5 = mm.median(noisy, 5)

mm.show(
    [img_gray_crop, noisy, f_gauss, f_media, f_median3, f_median5],
    titles=["Original", "Ruido 10%", "Gaussiano 5x5", "Media 5x5",
            "Mediana 3x3", "Mediana 5x5"],
    cols=3
)

## Aplicação Prática: Pré-processamento para Segmentação

Na prática, as técnicas deste capítulo raramente são usadas isoladamente. Um ***pipeline* de pré-processamento** típico combina várias etapas em sequência, adaptando-se ao tipo de imagem e à aplicação. A @fig-03-pipeline ilustra um *pipeline* completo:

1. **Equalização de histograma (CLAHE):** normaliza o contraste independente das condições de iluminação;
2. **Filtro Gaussiano:** suaviza ruído de aquisição sem destruir bordas;
3. **Detecção de bordas (Sobel/Canny):** extrai estruturas relevantes para segmentação.

::: {.callout-note}
### Ordem importa {.unnumbered}
A ordem das operações afeta o resultado final. Em geral: **(1) normalização de intensidade → (2) redução de ruído → (3) realce/segmentação**. Inverter a ordem pode amplificar ruído ou perder bordas antes de detectá-las.
:::

In [ ]:
#[py]#
#| label: fig-03-pipeline
#| fig-cap: "*Pipeline* de pré-processamento: CLAHE → Gaussiano → Canny. Cada etapa prepara a imagem para a seguinte, resultando em bordas limpas e bem definidas."
#| echo: true
#| output: true

# Etapa 1: CLAHE
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
img_clahe = clahe.apply(img_gray_crop)

# Etapa 2: Gaussiano
img_gauss = cv2.GaussianBlur(img_clahe, (5, 5), 0)

# Etapa 3: Canny
edges = cv2.Canny(img_gauss, 50, 150)

# Comparação: pipeline vs. Canny direto
edges_direct = cv2.Canny(img_gray_crop, 50, 150)

mm.show(
    [img_gray_crop, img_clahe, img_gauss, edges, edges_direct],
    titles=["Original", "1. CLAHE", "2. Gaussiano", "3. Canny (pipeline)", "Canny (direto)"],
    cols=5
)

In [ ]:
#[cpp]#
#| label: fig-03-pipeline
#| fig-cap: "*Pipeline* de pré-processamento: equalização → Gaussiano → Canny. A trilha Python usa CLAHE no lugar da equalização global; CLAHE não tem equivalente em morph.hpp."
#| echo: true
#| output: true

img_eq    = mm.equalize(img_gray_crop)     # Etapa 1: equalização global
img_gauss = mm.gaussian(img_eq, 5, 0)      # Etapa 2: Gaussiano
edges     = mm.canny(img_gauss, 50, 150)   # Etapa 3: Canny

edges_direct = mm.canny(img_gray_crop, 50, 150)   # Canny direto, sem pré-processo

mm.show(
    [img_gray_crop, img_eq, img_gauss, edges, edges_direct],
    titles=["Original", "1. Equalizado", "2. Gaussiano", "3. Canny (pipeline)", "Canny (direto)"],
    cols=5
)

## Resumo

Neste capítulo foram apresentadas as principais técnicas de processamento no domínio espacial, da manipulação direta de pixels até a filtragem por vizinhança:

- **Operações de ponto:** aritméticas saturadas (`mm.addm`, `mm.subm`) e lógicas bit a bit (`mm.band`, `mm.bor`, `mm.bnot`) para recorte de ROI e combinação de imagens; *alpha blending* (`mm.blend`) para fusão ponderada com peso $\alpha \in [0,1]$.
- **Histograma:** função discreta de distribuição de intensidades; visualizado com `mm.histImg` e calculado com `mm.hist`; base para diagnóstico tonal e para as técnicas de equalização e especificação.
- **Equalização:** redistribuição automática das intensidades pela CDF (`mm.equalize`), com variante adaptativa CLAHE para controle local do contraste.
- **Especificação de histograma:** transferência do perfil tonal de uma imagem de referência via mapeamento inverso da CDF — generalização da equalização para distribuições arbitrárias.
- **Correlação e convolução:** mecanismo de janela deslizante implementado em `mm.conv` (`cv2.filter2D`); diferenciados pela rotação de 180° do *kernel* — relevante apenas para kernels assimétricos.
- **Filtros de suavização:** média (*kernel* uniforme, borra bordas proporcionalmente ao tamanho) e Gaussiano (ponderação radial, separável, sem *ringing*, preserva melhor as bordas).
- **Filtros de realce:** Laplaciano ($w_4$/$w_8$, segunda derivada isotrópica), Sobel (gradiente direcional de primeira ordem, com magnitude $|\nabla f|$ e direção $\theta$) e *Unsharp Masking* (amplificação das altas frequências com parâmetro $k$).
- **Filtro da mediana:** não linear, robusto a outliers, preserva bordas — superior aos filtros lineares para ruído sal e pimenta.
- **Pipeline prático:** encadeamento CLAHE → Gaussiano → Canny como estratégia de pré-processamento; `mm.drawImgKernel` para visualização didática da janela deslizante.

O Capítulo 4 abordará a **morfologia matemática** (erosão, dilatação, abertura e fechamento), explorando em profundidade as funções `mm.ero` e `mm.dil` da biblioteca `morph.py`. Em seguida, o Capítulo 5 apresentará o **processamento no domínio da frequência**, com foco na Transformada de Fourier e em técnicas de filtragem espectral.

## 🤖 Uso do Gemini Notebook como Tutor Complementar

Nesta edição, incentivamos o uso do **Gemini Notebook** como ferramenta complementar de aprendizagem. Essa ferramenta de IA utiliza exclusivamente os documentos fornecidos pelo autor como base de conhecimento, garantindo respostas coerentes com o conteúdo do livro — incluindo as funções da biblioteca `morph.py` e os experimentos realizados neste capítulo.

Para cada capítulo, preparamos um projeto específico na plataforma com o PDF do capítulo, os *notebooks* e materiais auxiliares. Sugerimos explorar especialmente:

- **Guia de Estudo:** resumo estruturado dos conceitos, ideal para revisão antes de provas;
- **Conversa:** tire dúvidas sobre equalização, convolução, filtros e pipelines diretamente com o tutor;
- **Perguntas frequentes:** questões típicas sobre a diferença entre média e mediana, USM, Laplaciano vs. Sobel.

::: {.callout-important appearance="default" icon=false}
### 🎓 Estude com o Tutor Inteligente {.unnumbered}

Para interagir com o conteúdo deste capítulo, acesse o *link* a seguir. O ambiente contém materiais didáticos em diferentes formatos, gerados a partir do **PDF** do capítulo. Na plataforma, explore especialmente as opções **Guia de Estudo** e **Conversa** para aprofundar sua compreensão.

[🚀 ACESSAR Gemini Notebook: CAPÍTULO 03](https://notebooklm.google.com/notebook/d6593e26-a008-4d3b-8073-5c9b7d00eacc)

#### 🌐 Idioma e Linguagem de Programação {.unnumbered}

O projeto deste capítulo no Gemini Notebook foi construído apenas com o texto em **português** e os exemplos de código em **Python**. Se você está estudando pela edição em inglês ou francês, ou acompanhando a trilha em C++, as respostas do tutor podem não corresponder exatamente à versão que você está lendo.

#### ⚠️ Aviso sobre Conteúdo Gerado por IA {.unnumbered}

A IA é uma poderosa aliada nos estudos, mas o conteúdo gerado pode conter **erros ou imprecisões**. Consulte sempre **livros, artigos científicos e outras fontes acadêmicas confiáveis** para validar as informações. Sempre que possível, execute os exemplos práticos fornecidos neste capítulo para verificar os resultados.
:::

## Lista de Exercícios

1. **(10%)** Explique a diferença entre **convolução** e **correlação cruzada**. Para quais tipos de *kernel* os resultados são idênticos? Dê um exemplo de *kernel* assimétrico (como Sobel $G_x$) e mostre numericamente que os resultados diferem aplicando-o ao patch 5×5 do capítulo das duas formas.

2. **(15%)** Considere uma imagem 5×5 com intensidades concentradas entre os níveis 3 e 5 (baixo contraste, 3 bits). Aplique manualmente o algoritmo de equalização da @tbl-03-equalizacao, preenchendo todas as colunas da tabela ($k$, $h[k]$, $p[k]$, $\text{cdf}[k]$, $\text{lut}[k]$). Verifique o resultado com `mm.equalize`.

3. **(15%)** Usando `mm.conv`, aplique o filtro de média com kernels de tamanho 3×3, 9×9 e 21×21 à imagem do mandrill. Para cada versão, calcule o **PSNR** (*Peak Signal-to-Noise Ratio*) em relação à original:
$$\text{PSNR} = 10\log_{10}\!\left(\frac{255^2}{\text{MSE}}\right), \quad \text{MSE} = \frac{1}{MN}\sum_{i,j}(f-g)^2$$
Plote o PSNR em função do tamanho do kernel e explique o que a queda progressiva indica sobre a relação entre suavização e perda de informação.

4. **(15%)** Usando `add_salt_pepper` com densidade de 5%, aplique e compare: (a) `mm.conv` com média 3×3, (b) `cv2.GaussianBlur` com $\sigma=1$, (c) `cv2.medianBlur` com janela 3×3 e (d) `cv2.medianBlur` com janela 5×5. Exiba as imagens com `mm.show` em grade 2×4 (linha 1: imagens, linha 2: histogramas via `mm.histImg`). Explique por que a mediana supera os filtros lineares usando o argumento da @tbl-03-mediana.

5. **(15%)** Implemente `mm.conv0` usando apenas operações NumPy vetorizadas — sem laços Python e sem `cv2.filter2D` — com o operador de *stride tricks* (`np.lib.stride_tricks.sliding_window_view`). Compare o resultado e o tempo de execução com `mm.conv0` (laços) e `mm.conv` (cv2) para kernels 3×3 e 15×15 na imagem do mandrill.

6. **(15%)** Aplique o *Unsharp Masking* com $\sigma=1$ e $k \in \{0.5, 1.0, 2.0, 4.0\}$ usando a função `usm` do capítulo. Para cada valor de $k$: (a) calcule a diferença absoluta $|g - f|$, (b) exiba as imagens e as diferenças com `mm.show`, e (c) plote o histograma das diferenças com `mm.histImg`. Identifique a partir de qual $k$ os artefatos (halos e amplificação de ruído) tornam-se visualmente inaceitáveis.

7. **(15%)** Escolha uma imagem de raio-X ou tomografia disponível publicamente (ex.: via `mm.read` de URL) e projete um pipeline de pré-processamento com pelo menos 4 etapas sequenciais, justificando cada escolha com base nos conceitos do capítulo. Exiba com `mm.show` em grade: imagem original, cada etapa intermediária e o resultado final com seus histogramas (`mm.histImg`).

## Referências do Capítulo {.unnumbered}

A fundamentação teórica deste capítulo baseia-se nas seguintes obras:

* @gonzalez2018digital para os conceitos de operações de intensidade, histograma, convolução e filtragem espacial.
* @szeliski2022 para a visão computacional e aplicações práticas de filtragem.
* @bradski2008learning para a implementação prática com OpenCV e `morph.py`.